# EcoPackAI – Feature Engineering and Validation

## Objective
Prepare meaningful features for sustainable packaging recommendation.

### Tasks
1. Load cleaned datasets.
2. Validate input features and target variables.
3. Create derived sustainability and cost-efficiency features.
4. Check feature correlations and potential data leakage.
5. Prepare separate datasets for cost and CO2 prediction.

### ML Objectives
- Random Forest Regressor – Cost Prediction
- XGBoost Regressor – CO2 Footprint Prediction

In [1]:
# ==========================================
# 1. IMPORT REQUIRED LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# ==========================================
# 2. DEFINE PROJECT PATHS
# ==========================================

ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "processed"

# ==========================================
# 3. LOAD CLEANED DATASETS
# ==========================================

materials = pd.read_csv(DATA_DIR / "materials_cleaned.csv")

products = pd.read_csv(DATA_DIR / "products_cleaned.csv")

training = pd.read_csv(DATA_DIR / "training_data_cleaned.csv")

# ==========================================
# 4. DISPLAY DATASET INFORMATION
# ==========================================

print("ECOPACKAI - FEATURE ENGINEERING STARTED")

print("\nDataset Shapes:")
print("Materials:", materials.shape)
print("Products:", products.shape)
print("Training Data:", training.shape)

print("\nAll datasets loaded successfully!")

ECOPACKAI - FEATURE ENGINEERING STARTED

Dataset Shapes:
Materials: (120, 11)
Products: (300, 8)
Training Data: (36000, 22)

All datasets loaded successfully!


In [2]:
# ==========================================
# 5. INSPECT EXISTING TRAINING FEATURES
# ==========================================

print("TRAINING DATA COLUMNS")
print("=" * 60)

for column in training.columns:
    print(column)

print("\nTARGET VARIABLE SUMMARY")
print("=" * 60)

targets = [
    "cost_target_inr_per_pack",
    "co2_target_kgco2e_per_pack",
    "suitability_score",
    "suitable_label"
]

display(training[targets].describe().round(3))

print("\nFIRST FIVE RECORDS")
display(training.head())

TRAINING DATA COLUMNS
product_id
material_id
industry
product_weight_kg
fragility_score
moisture_sensitive
required_strength_score
budget_per_pack_inr
material_category
strength_score
max_weight_capacity_kg
biodegradability_score
recyclability_percent
co2_kg_per_kg_material
cost_inr_per_kg
moisture_resistance_score
durability_score
packaging_mass_kg_synthetic
cost_target_inr_per_pack
co2_target_kgco2e_per_pack
suitability_score
suitable_label

TARGET VARIABLE SUMMARY


,cost_target_inr_per_pack,co2_target_kgco2e_per_pack,suitability_score,suitable_label
count,36000.000,36000.000,36000.000,36000.000
mean,18.817,0.626,80.994,0.182
std,14.559,0.908,7.175,0.385
min,0.580,0.015,54.830,0.000
25%,7.508,0.181,75.790,0.000
50%,15.147,0.353,80.840,0.000
75%,26.607,0.671,86.260,0.000
max,70.685,6.221,95.650,1.000



FIRST FIVE RECORDS


,product_id,material_id,industry,product_weight_kg,fragility_score,moisture_sensitive,required_strength_score,budget_per_pack_inr,material_category,strength_score,...,recyclability_percent,co2_kg_per_kg_material,cost_inr_per_kg,moisture_resistance_score,durability_score,packaging_mass_kg_synthetic,cost_target_inr_per_pack,co2_target_kgco2e_per_pack,suitability_score,suitable_label
0,P0001,M0101,Food,2.12,39,1,46,12.44,Paper,58.1,...,86,0.834,31.66,26,37,0.159,5.6380,0.1326,84.00,0
1,P0001,M0102,Food,2.12,39,1,46,12.44,Paper,50.9,...,84,0.739,27.19,27,36,0.159,4.8420,0.1175,82.60,0
2,P0001,M0103,Food,2.12,39,1,46,12.44,Paper,51.5,...,91,0.800,29.46,33,42,0.159,5.2462,0.1272,86.15,0
3,P0001,M0104,Food,2.12,39,1,46,12.44,Paper,55.5,...,95,0.797,24.77,31,38,0.159,4.4110,0.1267,85.50,0
4,P0001,M0105,Food,2.12,39,1,46,12.44,Paper,48.3,...,86,0.819,25.85,26,45,0.159,4.6034,0.1302,84.50,0


In [3]:
# ==========================================
# 6. FEATURE RELATIONSHIP VALIDATION
# ==========================================

# Select numerical columns
numeric_data = training.select_dtypes(include=np.number)

# Correlation with target variables
target_columns = [
    "cost_target_inr_per_pack",
    "co2_target_kgco2e_per_pack",
    "suitability_score",
    "suitable_label"
]

print("CORRELATION WITH TARGET VARIABLES")
print("=" * 60)

correlations = numeric_data.corr()[target_columns]

display(correlations.round(3))

# Check possible CO2 formula relationship
training["calculated_co2_component"] = (
    training["packaging_mass_kg_synthetic"]
    * training["co2_kg_per_kg_material"]
)

print("\nCO2 TARGET VS CALCULATED COMPONENT")
print("=" * 60)

display(
    training[
        [
            "packaging_mass_kg_synthetic",
            "co2_kg_per_kg_material",
            "calculated_co2_component",
            "co2_target_kgco2e_per_pack"
        ]
    ].head(10).round(4)
)

# Check whether the suitability score and label agree
print("\nSUITABILITY SCORE AND LABEL RELATIONSHIP")
print("=" * 60)

display(
    training.groupby("suitable_label")["suitability_score"]
    .agg(["count", "mean", "min", "max"])
    .round(2)
)

CORRELATION WITH TARGET VARIABLES


,cost_target_inr_per_pack,co2_target_kgco2e_per_pack,suitability_score,suitable_label
product_weight_kg,0.739,0.394,-0.341,-0.483
fragility_score,0.059,0.031,-0.028,-0.031
moisture_sensitive,-0.046,-0.025,-0.248,-0.123
required_strength_score,0.006,0.003,-0.137,-0.156
budget_per_pack_inr,-0.075,-0.040,0.304,0.280
strength_score,0.478,0.462,-0.280,0.076
max_weight_capacity_kg,0.493,0.577,-0.283,0.040
biodegradability_score,-0.541,-0.480,0.570,-0.051
recyclability_percent,-0.193,0.025,0.577,-0.034
co2_kg_per_kg_material,0.371,0.798,-0.267,0.013



CO2 TARGET VS CALCULATED COMPONENT


,packaging_mass_kg_synthetic,co2_kg_per_kg_material,calculated_co2_component,co2_target_kgco2e_per_pack
0,0.159,0.834,0.1326,0.1326
1,0.159,0.739,0.1175,0.1175
2,0.159,0.800,0.1272,0.1272
3,0.159,0.797,0.1267,0.1267
4,0.159,0.819,0.1302,0.1302
5,0.159,0.662,0.1053,0.1053
6,0.159,0.800,0.1272,0.1272
7,0.159,0.754,0.1199,0.1199
8,0.159,0.736,0.1170,0.1170
9,0.159,0.763,0.1213,0.1213



SUITABILITY SCORE AND LABEL RELATIONSHIP


,count,mean,min,max
suitable_label,,,,
0,29466,79.77,54.83,95.62
1,6534,86.53,74.00,95.65


In [4]:
# ==========================================
# 7. VERIFY SYNTHETIC TARGET FORMULAS
# ==========================================

# CO2 formula verification
training["calculated_co2"] = (
    training["packaging_mass_kg_synthetic"]
    * training["co2_kg_per_kg_material"]
)

co2_difference = (
    training["co2_target_kgco2e_per_pack"]
    - training["calculated_co2"]
).abs()

print("CO2 FORMULA VALIDATION")
print("Maximum difference:", co2_difference.max())
print("Matching records:", (co2_difference < 0.00001).sum())
print("Total records:", len(training))

# Cost formula investigation
training["calculated_cost"] = (
    training["packaging_mass_kg_synthetic"]
    * training["cost_inr_per_kg"]
)

cost_difference = (
    training["cost_target_inr_per_pack"]
    - training["calculated_cost"]
).abs()

print("\nCOST FORMULA INVESTIGATION")
print("Maximum difference:", cost_difference.max())
print("Mean difference:", cost_difference.mean())
print("Matching records:", (cost_difference < 0.0001).sum())
print("Total records:", len(training))

CO2 FORMULA VALIDATION
Maximum difference: 0.0005661999999997391
Matching records: 5606
Total records: 36000

COST FORMULA INVESTIGATION
Maximum difference: 7.5733749999999915
Mean difference: 2.016227888333333
Matching records: 0
Total records: 36000


In [5]:
# ==========================================
# 8. DETAILED TARGET FORMULA INVESTIGATION
# ==========================================

# A. CO2 validation with rounding tolerance

print("CO2 ROUNDING VALIDATION")
print("=" * 55)

for tolerance in [0.0001, 0.0005, 0.001]:
    matches = (co2_difference <= tolerance).sum()

    print(
        f"Tolerance {tolerance}: "
        f"{matches}/{len(training)} records"
    )

# B. Cost adjustment investigation

training["cost_base"] = (
    training["packaging_mass_kg_synthetic"]
    * training["cost_inr_per_kg"]
)

training["cost_adjustment"] = (
    training["cost_target_inr_per_pack"]
    - training["cost_base"]
)

training["cost_multiplier"] = (
    training["cost_target_inr_per_pack"]
    / training["cost_base"]
)

print("\nCOST ADJUSTMENT SUMMARY")
print("=" * 55)

display(
    training[
        [
            "cost_base",
            "cost_target_inr_per_pack",
            "cost_adjustment",
            "cost_multiplier"
        ]
    ].describe().round(4)
)

# C. Inspect first ten records

print("\nCOST FORMULA SAMPLE")
display(
    training[
        [
            "product_id",
            "material_id",
            "packaging_mass_kg_synthetic",
            "cost_inr_per_kg",
            "cost_base",
            "cost_target_inr_per_pack",
            "cost_adjustment",
            "cost_multiplier"
        ]
    ].head(10).round(4)
)

CO2 ROUNDING VALIDATION
Tolerance 0.0001: 31216/36000 records
Tolerance 0.0005: 35597/36000 records
Tolerance 0.001: 36000/36000 records

COST ADJUSTMENT SUMMARY


,cost_base,cost_target_inr_per_pack,cost_adjustment,cost_multiplier
count,36000.0000,36000.0000,36000.0000,36000.0000
mean,16.8004,18.8166,2.0162,1.1200
std,12.9992,14.5593,1.5601,0.0003
min,0.5180,0.5802,0.0622,1.1179
25%,6.7039,7.5075,0.8047,1.1200
50%,13.5231,15.1474,1.6226,1.1200
75%,23.7549,26.6069,2.8507,1.1201
max,63.1118,70.6852,7.5734,1.1219



COST FORMULA SAMPLE


,product_id,material_id,packaging_mass_kg_synthetic,cost_inr_per_kg,cost_base,cost_target_inr_per_pack,cost_adjustment,cost_multiplier
0,P0001,M0101,0.159,31.66,5.0339,5.6380,0.6041,1.12
1,P0001,M0102,0.159,27.19,4.3232,4.8420,0.5188,1.12
2,P0001,M0103,0.159,29.46,4.6841,5.2462,0.5621,1.12
3,P0001,M0104,0.159,24.77,3.9384,4.4110,0.4726,1.12
4,P0001,M0105,0.159,25.85,4.1102,4.6034,0.4932,1.12
5,P0001,M0106,0.159,25.94,4.1245,4.6194,0.4949,1.12
6,P0001,M0107,0.159,31.43,4.9974,5.5971,0.5997,1.12
7,P0001,M0108,0.159,25.54,4.0609,4.5482,0.4873,1.12
8,P0001,M0109,0.159,28.99,4.6094,5.1625,0.5531,1.12
9,P0001,M0110,0.159,24.43,3.8844,4.3505,0.4661,1.12


In [6]:
# ==========================================
# 9. FINAL SYNTHETIC FORMULA VERIFICATION
# ==========================================

# COST FORMULA
training["calculated_cost"] = (
    training["packaging_mass_kg_synthetic"]
    * training["cost_inr_per_kg"]
    * 1.12
)

training["cost_formula_difference"] = abs(
    training["cost_target_inr_per_pack"]
    - training["calculated_cost"]
)

# CO2 FORMULA
training["calculated_co2"] = (
    training["packaging_mass_kg_synthetic"]
    * training["co2_kg_per_kg_material"]
)

training["co2_formula_difference"] = abs(
    training["co2_target_kgco2e_per_pack"]
    - training["calculated_co2"]
)

print("=" * 55)
print("FINAL SYNTHETIC FORMULA VERIFICATION")
print("=" * 55)

print("\nCOST FORMULA")
print("Formula: Packaging Mass x Material Cost x 1.12")
print("Maximum difference:",
      training["cost_formula_difference"].max())
print("Mean difference:",
      training["cost_formula_difference"].mean())
print("Records within 0.001:",
      (training["cost_formula_difference"] <= 0.001).sum())

print("\nCO2 FORMULA")
print("Formula: Packaging Mass x CO2 Emission Factor")
print("Maximum difference:",
      training["co2_formula_difference"].max())
print("Mean difference:",
      training["co2_formula_difference"].mean())
print("Records within 0.001:",
      (training["co2_formula_difference"] <= 0.001).sum())

FINAL SYNTHETIC FORMULA VERIFICATION

COST FORMULA
Formula: Packaging Mass x Material Cost x 1.12
Maximum difference: 0.005955200000009597
Mean difference: 0.0013874939333330001
Records within 0.001: 19920

CO2 FORMULA
Formula: Packaging Mass x CO2 Emission Factor
Maximum difference: 0.0005661999999997391
Mean difference: 5.972617777777183e-05
Records within 0.001: 36000


In [7]:
# COST FORMULA DIFFERENCE INVESTIGATION

print("COST DIFFERENCE DISTRIBUTION")
print(training["cost_formula_difference"].describe())

print("\nRECORDS WITH LARGEST COST DIFFERENCES")

display(
    training.nlargest(
        10,
        "cost_formula_difference"
    )[
        [
            "product_id",
            "material_id",
            "packaging_mass_kg_synthetic",
            "cost_inr_per_kg",
            "cost_target_inr_per_pack",
            "calculated_cost",
            "cost_formula_difference"
        ]
    ]
)

COST DIFFERENCE DISTRIBUTION
count    36000.000000
mean         0.001387
std          0.001779
min          0.000000
25%          0.000022
50%          0.000044
75%          0.002395
max          0.005955
Name: cost_formula_difference, dtype: float64

RECORDS WITH LARGEST COST DIFFERENCES


,product_id,material_id,packaging_mass_kg_synthetic,cost_inr_per_kg,cost_target_inr_per_pack,calculated_cost,cost_formula_difference
7062,P0059,M1103,0.4688,105.45,55.3612,55.367155,0.005955
23622,P0197,M1103,0.2438,105.45,28.7878,28.793755,0.005955
462,P0004,M1103,0.0562,105.45,6.6434,6.637445,0.005955
4662,P0039,M1103,0.3562,105.45,42.0746,42.068645,0.005955
3942,P0033,M1103,0.5812,105.45,68.6480,68.642045,0.005955
31182,P0260,M1103,0.2678,105.45,31.6223,31.628251,0.005951
17622,P0147,M1103,0.4822,105.45,56.9557,56.949749,0.005951
34782,P0290,M1103,0.0668,105.45,7.8834,7.889347,0.005947
9942,P0083,M1103,0.0832,105.45,9.8322,9.826253,0.005947
18102,P0151,M1103,0.1957,105.45,23.1189,23.112953,0.005947


In [8]:
# COST FORMULA ERROR BY MATERIAL

material_error = (
    training.groupby("material_id")
    ["cost_formula_difference"]
    .agg(["count", "mean", "max"])
    .sort_values("mean", ascending=False)
)

display(material_error.head(15).round(6))

,count,mean,max
material_id,,,
M1103,300,0.002652,0.005955
M1207,300,0.002590,0.005826
M1101,300,0.002580,0.005788
M1105,300,0.002572,0.005777
M1209,300,0.002519,0.005657
M1202,300,0.002507,0.005628
M1201,300,0.002501,0.005613
M1208,300,0.002412,0.005419
M1108,300,0.002408,0.005413


In [9]:
# CHECK M1103 MATERIAL DETAILS

display(
    materials[materials["material_id"] == "M1103"]
)

,material_id,material_name,material_category,strength_score,max_weight_capacity_kg,biodegradability_score,recyclability_percent,co2_kg_per_kg_material,cost_inr_per_kg,moisture_resistance_score,durability_score
102,M1103,Glass - Variant 03,Glass,87.9,28.38,7,97,1.138,105.45,89,100


In [10]:
training["cost_relative_error_percent"] = (
    training["cost_formula_difference"]
    / training["cost_target_inr_per_pack"]
) * 100

print("Mean relative error (%):",
      training["cost_relative_error_percent"].mean().round(6))

print("Maximum relative error (%):",
      training["cost_relative_error_percent"].max().round(6))

print("Records with relative error below 0.1%:",
      (training["cost_relative_error_percent"] < 0.1).sum(),
      "/", len(training))

Mean relative error (%): 0.012093
Maximum relative error (%): 0.186247
Records with relative error below 0.1%: 35280 / 36000


In [11]:
display(
    training[
        [
            "material_id",
            "product_id",
            "suitability_score",
            "suitable_label",
            "co2_target_kgco2e_per_pack",
            "cost_target_inr_per_pack"
        ]
    ].head(10)
)

display(
    training["suitability_score"].describe().round(3)
)

,material_id,product_id,suitability_score,suitable_label,co2_target_kgco2e_per_pack,cost_target_inr_per_pack
0,M0101,P0001,84.00,0,0.1326,5.6380
1,M0102,P0001,82.60,0,0.1175,4.8420
2,M0103,P0001,86.15,0,0.1272,5.2462
3,M0104,P0001,85.50,0,0.1267,4.4110
4,M0105,P0001,84.50,0,0.1302,4.6034
5,M0106,P0001,84.35,0,0.1053,4.6194
6,M0107,P0001,85.50,0,0.1272,5.5971
7,M0108,P0001,84.15,0,0.1199,4.5482
8,M0109,P0001,84.45,0,0.1170,5.1625
9,M0110,P0001,85.20,0,0.1213,4.3505


count    36000.000
mean        80.994
std          7.175
min         54.830
25%         75.790
50%         80.840
75%         86.260
max         95.650
Name: suitability_score, dtype: float64

In [12]:
data_dictionary = pd.read_csv(
    ROOT / "data" / "data_dictionary.csv"
)

display(
    data_dictionary[
        data_dictionary["column_name"].str.contains(
            "suitability", case=False, na=False
        )
    ]
)

,column_name,meaning,unit_or_scale
22,suitability_score,Rule-generated material ranking score,0–100 score


In [13]:
suitability_analysis = training[
    [
        "material_id",
        "strength_score",
        "biodegradability_score",
        "recyclability_percent",
        "moisture_resistance_score",
        "durability_score",
        "suitability_score"
    ]
]

display(suitability_analysis.head(15))

display(
    suitability_analysis.corr(numeric_only=True)
    ["suitability_score"]
    .sort_values(ascending=False)
)

,material_id,strength_score,biodegradability_score,recyclability_percent,moisture_resistance_score,durability_score,suitability_score
0,M0101,58.1,94,86,26,37,84.00
1,M0102,50.9,86,84,27,36,82.60
2,M0103,51.5,97,91,33,42,86.15
3,M0104,55.5,94,95,31,38,85.50
4,M0105,48.3,92,86,26,45,84.50
5,M0106,54.6,99,83,31,31,84.35
6,M0107,46.7,98,84,29,45,85.50
7,M0108,53.0,89,89,33,36,84.15
8,M0109,47.9,91,81,33,44,84.45
9,M0110,55.3,98,95,25,35,85.20


suitability_score            1.000000
recyclability_percent        0.577313
biodegradability_score       0.570167
strength_score              -0.280095
durability_score            -0.294270
moisture_resistance_score   -0.497590
Name: suitability_score, dtype: float64

In [14]:
score_correlations = (
    suitability_analysis
    .corr(numeric_only=True)["suitability_score"]
    .drop("suitability_score")
    .sort_values(ascending=False)
)

display(score_correlations.to_frame("correlation"))

,correlation
recyclability_percent,0.577313
biodegradability_score,0.570167
strength_score,-0.280095
durability_score,-0.294270
moisture_resistance_score,-0.497590


In [15]:
material_scores = (
    training.groupby("material_id")
    .agg(
        suitability_score=("suitability_score", "mean"),
        strength_score=("strength_score", "first"),
        biodegradability_score=("biodegradability_score", "first"),
        recyclability_percent=("recyclability_percent", "first"),
        moisture_resistance_score=("moisture_resistance_score", "first"),
        durability_score=("durability_score", "first")
    )
)

print("TOP 10 MATERIALS")
display(
    material_scores.sort_values(
        "suitability_score", ascending=False
    ).head(10).round(2)
)

print("BOTTOM 10 MATERIALS")
display(
    material_scores.sort_values(
        "suitability_score", ascending=True
    ).head(10).round(2)
)

TOP 10 MATERIALS


,suitability_score,strength_score,biodegradability_score,recyclability_percent,moisture_resistance_score,durability_score
material_id,,,,,,
M0503,89.95,70.2,99,97,39,61
M0307,89.62,68.6,99,98,27,55
M0505,89.39,67.6,100,92,47,58
M0602,89.25,76.1,93,94,42,67
M0504,89.01,62.7,100,97,44,56
M0610,89.00,75.8,95,85,42,68
M0509,88.49,66.8,98,94,40,61
M0601,88.43,74.3,93,84,51,70
M0607,88.30,78.3,92,81,48,68


BOTTOM 10 MATERIALS


,suitability_score,strength_score,biodegradability_score,recyclability_percent,moisture_resistance_score,durability_score
material_id,,,,,,
M1010,70.01,86.0,6,23,78,80
M1004,70.45,90.3,11,30,77,71
M1001,71.06,86.9,7,36,80,76
M1009,71.15,85.3,11,28,84,79
M1008,71.42,81.9,15,24,86,73
M1002,71.45,80.1,19,28,77,74
M1006,71.58,78.3,7,36,78,80
M1005,71.60,83.9,13,27,77,81
M1003,72.07,88.7,14,33,78,80


In [16]:
score_consistency = (
    training.groupby("material_id")
    ["suitability_score"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("std", ascending=False)
)

display(score_consistency.head(10).round(3))

,mean,std,min,max
material_id,,,,
M0105,79.250,7.519,57.02,91.90
M0107,80.204,7.269,58.78,92.60
M0103,81.749,6.980,60.60,92.85
M0109,79.740,6.913,58.77,91.15
M0104,81.875,6.899,60.49,92.40
M0102,79.418,6.662,58.55,89.90
M0702,73.524,6.640,54.83,84.85
M0106,81.288,6.599,60.53,91.25
M0108,80.901,6.526,60.24,90.85


In [17]:
category_analysis = (
    training.groupby("material_category")
    ["suitability_score"]
    .agg(["mean", "std", "min", "max", "count"])
    .sort_values("mean", ascending=False)
)

display(category_analysis.round(2))

,mean,std,min,max,count
material_category,,,,,
Plant Fiber,87.68,4.62,70.46,95.65,6000
Paper,84.75,6.06,57.02,95.15,12000
Glass,79.25,3.47,71.34,85.85,3000
Metal,79.06,3.41,72.84,85.45,3000
Bioplastic,76.02,6.02,54.83,86.90,3000
Plastic,74.41,4.51,61.44,84.80,9000


In [18]:
co2_analysis = (
    materials.groupby("material_category")
    ["co2_kg_per_kg_material"]
    .agg(["mean", "min", "max", "std", "count"])
    .sort_values("mean")
)

display(co2_analysis.round(4))

,mean,min,max,std,count
material_category,,,,,
Plant Fiber,0.7960,0.654,0.983,0.0824,20
Paper,0.8066,0.616,1.055,0.0983,40
Glass,1.2120,1.106,1.292,0.0675,10
Bioplastic,1.8819,1.628,2.067,0.1624,10
Plastic,2.3617,2.070,2.730,0.1912,30
Metal,9.5567,8.719,10.394,0.6362,10


In [19]:
# Calculate CO2 Impact Index
max_co2 = materials["co2_kg_per_kg_material"].max()

materials["co2_impact_index"] = (
    materials["co2_kg_per_kg_material"] / max_co2
) * 100

# Inspect the results
display(
    materials[
        [
            "material_id",
            "material_name",
            "material_category",
            "co2_kg_per_kg_material",
            "co2_impact_index"
        ]
    ].sort_values("co2_impact_index").head(15).round(3)
)

# Summary statistics
display(
    materials["co2_impact_index"].describe().round(3)
)

,material_id,material_name,material_category,co2_kg_per_kg_material,co2_impact_index
20,M0301,Molded Pulp,Paper,0.616,5.926
40,M0501,Bagasse Fiber,Plant Fiber,0.654,6.292
5,M0106,Kraft Paper - Variant 06,Paper,0.662,6.369
26,M0307,Molded Pulp - Variant 07,Paper,0.666,6.408
21,M0302,Molded Pulp - Variant 02,Paper,0.671,6.456
25,M0306,Molded Pulp - Variant 06,Paper,0.683,6.571
28,M0309,Molded Pulp - Variant 09,Paper,0.684,6.581
43,M0504,Bagasse Fiber - Variant 04,Plant Fiber,0.688,6.619
24,M0305,Molded Pulp - Variant 05,Paper,0.693,6.667
45,M0506,Bagasse Fiber - Variant 06,Plant Fiber,0.695,6.687


count    120.000
mean      19.686
std       22.871
min        5.926
25%        7.730
50%       10.395
75%       21.618
max      100.000
Name: co2_impact_index, dtype: float64

In [20]:
print("Total materials:", len(materials))

print("Missing index values:",
      materials["co2_impact_index"].isna().sum())

print("Minimum index:",
      materials["co2_impact_index"].min())

print("Maximum index:",
      materials["co2_impact_index"].max())

print("\nHighest CO2 impact materials:")

display(
    materials[
        ["material_id", "material_name",
         "co2_kg_per_kg_material", "co2_impact_index"]
    ]
    .sort_values("co2_impact_index", ascending=False)
    .head(10)
    .round(3)
)

Total materials: 120
Missing index values: 0
Minimum index: 5.926496055416586
Maximum index: 100.0

Highest CO2 impact materials:


,material_id,material_name,co2_kg_per_kg_material,co2_impact_index
111,M1202,Aluminium - Variant 02,10.394,100.000
114,M1205,Aluminium - Variant 05,10.145,97.604
118,M1209,Aluminium - Variant 09,10.075,96.931
119,M1210,Aluminium - Variant 10,10.073,96.912
110,M1201,Aluminium,9.979,96.007
117,M1208,Aluminium - Variant 08,9.229,88.792
116,M1207,Aluminium - Variant 07,9.189,88.407
115,M1206,Aluminium - Variant 06,8.999,86.579
113,M1204,Aluminium - Variant 04,8.765,84.327
112,M1203,Aluminium - Variant 03,8.719,83.885


In [21]:
# Calculate Cost Efficiency Index

min_cost = materials["cost_inr_per_kg"].min()

materials["cost_efficiency_index"] = (
    min_cost / materials["cost_inr_per_kg"]
) * 100

# Display the 15 most cost-efficient materials

display(
    materials[
        [
            "material_id",
            "material_name",
            "material_category",
            "cost_inr_per_kg",
            "cost_efficiency_index"
        ]
    ]
    .sort_values("cost_efficiency_index", ascending=False)
    .head(15)
    .round(3)
)

# Summary statistics

display(
    materials["cost_efficiency_index"].describe().round(3)
)

,material_id,material_name,material_category,cost_inr_per_kg,cost_efficiency_index
26,M0307,Molded Pulp - Variant 07,Paper,20.72,100.000
29,M0310,Molded Pulp - Variant 10,Paper,21.40,96.822
28,M0309,Molded Pulp - Variant 09,Paper,21.89,94.655
27,M0308,Molded Pulp - Variant 08,Paper,23.11,89.658
20,M0301,Molded Pulp,Paper,23.51,88.133
21,M0302,Molded Pulp - Variant 02,Paper,23.77,87.169
22,M0303,Molded Pulp - Variant 03,Paper,23.78,87.132
9,M0110,Kraft Paper - Variant 10,Paper,24.43,84.814
3,M0104,Kraft Paper - Variant 04,Paper,24.77,83.650
23,M0304,Molded Pulp - Variant 04,Paper,24.95,83.046


count    120.000
mean      46.923
std       21.968
min       19.649
25%       26.416
50%       43.091
75%       65.414
max      100.000
Name: cost_efficiency_index, dtype: float64

In [22]:
print("Total materials:", len(materials))

print("Missing index values:",
      materials["cost_efficiency_index"].isna().sum())

print("Minimum index:",
      materials["cost_efficiency_index"].min())

print("Maximum index:",
      materials["cost_efficiency_index"].max())

print("\nLeast cost-efficient materials:")

display(
    materials[
        [
            "material_id",
            "material_name",
            "cost_inr_per_kg",
            "cost_efficiency_index"
        ]
    ]
    .sort_values("cost_efficiency_index")
    .head(10)
    .round(3)
)

display(
    materials["cost_efficiency_index"].describe().round(3)
)

Total materials: 120
Missing index values: 0
Minimum index: 19.64912280701754
Maximum index: 100.0

Least cost-efficient materials:


,material_id,material_name,cost_inr_per_kg,cost_efficiency_index
102,M1103,Glass - Variant 03,105.45,19.649
116,M1207,Aluminium - Variant 07,103.15,20.087
100,M1101,Glass,102.47,20.221
104,M1105,Glass - Variant 05,102.28,20.258
118,M1209,Aluminium - Variant 09,100.13,20.693
111,M1202,Aluminium - Variant 02,99.62,20.799
110,M1201,Aluminium,99.34,20.858
117,M1208,Aluminium - Variant 08,95.88,21.610
107,M1108,Glass - Variant 08,95.79,21.631
113,M1204,Aluminium - Variant 04,95.20,21.765


count    120.000
mean      46.923
std       21.968
min       19.649
25%       26.416
50%       43.091
75%       65.414
max      100.000
Name: cost_efficiency_index, dtype: float64

In [23]:
suitability_features = [
    "strength_score",
    "biodegradability_score",
    "recyclability_percent",
    "moisture_resistance_score",
    "durability_score"
]

display(
    materials[suitability_features]
    .describe()
    .round(2)
)

,strength_score,biodegradability_score,recyclability_percent,moisture_resistance_score,durability_score
count,120.00,120.00,120.00,120.00,120.00
mean,74.26,57.35,75.81,58.27,66.30
std,14.24,39.94,23.00,24.67,21.06
min,46.70,0.00,23.00,21.00,31.00
25%,62.60,15.00,64.00,38.00,46.75
50%,74.70,81.50,85.00,51.00,64.50
75%,85.48,92.25,93.25,79.25,83.00
max,100.00,100.00,100.00,100.00,100.00


In [24]:
display(
    materials[suitability_features]
    .describe()
    .T
    .round(2)
)

,count,mean,std,min,25%,50%,75%,max
strength_score,120.0,74.26,14.24,46.7,62.60,74.7,85.48,100.0
biodegradability_score,120.0,57.35,39.94,0.0,15.00,81.5,92.25,100.0
recyclability_percent,120.0,75.81,23.00,23.0,64.00,85.0,93.25,100.0
moisture_resistance_score,120.0,58.27,24.67,21.0,38.00,51.0,79.25,100.0
durability_score,120.0,66.30,21.06,31.0,46.75,64.5,83.00,100.0


In [25]:
# Calculate material-level suitability score

materials["material_suitability_score"] = (
    materials["strength_score"]
    + materials["biodegradability_score"]
    + materials["recyclability_percent"]
    + materials["moisture_resistance_score"]
    + materials["durability_score"]
) / 5

# Display top 10 materials

display(
    materials[
        [
            "material_id",
            "material_name",
            "material_category",
            "material_suitability_score"
        ]
    ]
    .sort_values(
        "material_suitability_score",
        ascending=False
    )
    .head(10)
    .round(2)
)

,material_id,material_name,material_category,material_suitability_score
111,M1202,Aluminium - Variant 02,Metal,80.20
116,M1207,Aluminium - Variant 07,Metal,79.82
113,M1204,Aluminium - Variant 04,Metal,79.80
109,M1110,Glass - Variant 10,Glass,79.68
118,M1209,Aluminium - Variant 09,Metal,79.34
112,M1203,Aluminium - Variant 03,Metal,79.18
103,M1104,Glass - Variant 04,Glass,79.04
108,M1109,Glass - Variant 09,Glass,78.48
101,M1102,Glass - Variant 02,Glass,77.78
110,M1201,Aluminium,Metal,77.60


In [26]:
display(
    materials["material_suitability_score"]
    .describe()
    .round(2)
)

print(
    "Missing values:",
    materials["material_suitability_score"].isna().sum()
)

count    120.00
mean      66.40
std        7.27
min       53.08
25%       61.33
50%       65.30
75%       71.94
max       80.20
Name: material_suitability_score, dtype: float64

Missing values: 0


In [27]:
# Validate all three engineered features

engineered_features = [
    "co2_impact_index",
    "cost_efficiency_index",
    "material_suitability_score"
]

print("Total materials:", len(materials))

print("\nMissing values:")
display(materials[engineered_features].isna().sum())

print("\nFeature statistics:")
display(materials[engineered_features].describe().round(3))

print("\nChecking score ranges:")

for feature in engineered_features:
    print(
        feature,
        "Minimum:", round(materials[feature].min(), 3),
        "Maximum:", round(materials[feature].max(), 3),
        "Valid range:",
        materials[feature].between(0, 100).all()
    )

Total materials: 120

Missing values:


co2_impact_index              0
cost_efficiency_index         0
material_suitability_score    0
dtype: int64


Feature statistics:


,co2_impact_index,cost_efficiency_index,material_suitability_score
count,120.000,120.000,120.000
mean,19.686,46.923,66.398
std,22.871,21.968,7.271
min,5.926,19.649,53.080
25%,7.730,26.416,61.330
50%,10.395,43.091,65.300
75%,21.618,65.414,71.945
max,100.000,100.000,80.200



Checking score ranges:
co2_impact_index Minimum: 5.926 Maximum: 100.0 Valid range: True
cost_efficiency_index Minimum: 19.649 Maximum: 100.0 Valid range: True
material_suitability_score Minimum: 53.08 Maximum: 80.2 Valid range: True


In [28]:
# Save engineered material dataset

OUTPUT_DIR = ROOT / "data" / "processed"

engineered_materials_path = (
    OUTPUT_DIR / "materials_feature_engineered.csv"
)

materials.to_csv(
    engineered_materials_path,
    index=False
)

print("File saved successfully:")
print(engineered_materials_path)

print("Dataset shape:", materials.shape)

File saved successfully:
c:\Users\gouth\OneDrive\Pictures\Documents\EcoPackAI_Project\data\processed\materials_feature_engineered.csv
Dataset shape: (120, 14)


In [29]:
saved_materials = pd.read_csv(
    engineered_materials_path
)

print("Saved dataset shape:", saved_materials.shape)

print("\nNew engineered columns:")
display(
    saved_materials[
        [
            "material_id",
            "co2_impact_index",
            "cost_efficiency_index",
            "material_suitability_score"
        ]
    ].head(10)
)

Saved dataset shape: (120, 14)

New engineered columns:


,material_id,co2_impact_index,cost_efficiency_index,material_suitability_score
0,M0101,8.023860,65.445357,60.22
1,M0102,7.109871,76.204487,56.78
2,M0103,7.696748,70.332654,62.90
3,M0104,7.667885,83.649576,62.70
4,M0105,7.879546,80.154739,59.46
5,M0106,6.369059,79.876638,59.72
6,M0107,7.696748,65.924276,60.54
7,M0108,7.254185,81.127643,60.00
8,M0109,7.081008,71.472922,59.38
9,M0110,7.340774,84.813754,61.66


In [30]:
# Select material ID and engineered features

material_features = saved_materials[
    [
        "material_id",
        "co2_impact_index",
        "cost_efficiency_index",
        "material_suitability_score"
    ]
]

# Merge with training dataset

training_engineered = training.merge(
    material_features,
    on="material_id",
    how="left",
    validate="many_to_one"
)

print("Original training shape:", training.shape)
print("Engineered training shape:", training_engineered.shape)

print("\nMissing values in new features:")
display(
    training_engineered[
        [
            "co2_impact_index",
            "cost_efficiency_index",
            "material_suitability_score"
        ]
    ].isna().sum()
)

Original training shape: (36000, 31)
Engineered training shape: (36000, 34)

Missing values in new features:


co2_impact_index              0
cost_efficiency_index         0
material_suitability_score    0
dtype: int64

In [31]:
# Save integrated training dataset

integrated_path = (
    ROOT / "data" / "processed"
    / "training_data_feature_engineered.csv"
)

training_engineered.to_csv(
    integrated_path,
    index=False
)

print("Integrated dataset saved successfully!")
print("File:", integrated_path)
print("Shape:", training_engineered.shape)

Integrated dataset saved successfully!
File: c:\Users\gouth\OneDrive\Pictures\Documents\EcoPackAI_Project\data\processed\training_data_feature_engineered.csv
Shape: (36000, 34)


In [32]:
verified_training = pd.read_csv(integrated_path)

print("Verified shape:", verified_training.shape)

display(
    verified_training[
        [
            "product_id",
            "material_id",
            "co2_impact_index",
            "cost_efficiency_index",
            "material_suitability_score"
        ]
    ].head(10)
)

Verified shape: (36000, 34)


,product_id,material_id,co2_impact_index,cost_efficiency_index,material_suitability_score
0,P0001,M0101,8.023860,65.445357,60.22
1,P0001,M0102,7.109871,76.204487,56.78
2,P0001,M0103,7.696748,70.332654,62.90
3,P0001,M0104,7.667885,83.649576,62.70
4,P0001,M0105,7.879546,80.154739,59.46
5,P0001,M0106,6.369059,79.876638,59.72
6,P0001,M0107,7.696748,65.924276,60.54
7,P0001,M0108,7.254185,81.127643,60.00
8,P0001,M0109,7.081008,71.472922,59.38
9,P0001,M0110,7.340774,84.813754,61.66


In [33]:
# Identify categorical columns
categorical_columns = (
    training_engineered
    .select_dtypes(include=["object", "category"])
    .columns.tolist()
)

# Identify numerical columns
numerical_columns = (
    training_engineered
    .select_dtypes(include=["number"])
    .columns.tolist()
)

print("Categorical columns:")
print(categorical_columns)

print("\nNumber of categorical columns:",
      len(categorical_columns))

print("\nNumerical columns:")
print(numerical_columns)

print("\nNumber of numerical columns:",
      len(numerical_columns))

Categorical columns:
['product_id', 'material_id', 'industry', 'material_category']

Number of categorical columns: 4

Numerical columns:
['product_weight_kg', 'fragility_score', 'moisture_sensitive', 'required_strength_score', 'budget_per_pack_inr', 'strength_score', 'max_weight_capacity_kg', 'biodegradability_score', 'recyclability_percent', 'co2_kg_per_kg_material', 'cost_inr_per_kg', 'moisture_resistance_score', 'durability_score', 'packaging_mass_kg_synthetic', 'cost_target_inr_per_pack', 'co2_target_kgco2e_per_pack', 'suitability_score', 'suitable_label', 'calculated_co2_component', 'calculated_co2', 'calculated_cost', 'cost_base', 'cost_adjustment', 'cost_multiplier', 'cost_formula_difference', 'co2_formula_difference', 'cost_relative_error_percent', 'co2_impact_index', 'cost_efficiency_index', 'material_suitability_score']

Number of numerical columns: 30


C:\Users\gouth\AppData\Local\Temp\ipykernel_31676\864851738.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category"])


In [34]:
# Temporary validation columns

diagnostic_columns = [
    "calculated_co2_component",
    "calculated_co2",
    "calculated_cost",
    "cost_base",
    "cost_adjustment",
    "cost_multiplier",
    "cost_formula_difference",
    "co2_formula_difference",
    "cost_relative_error_percent"
]

# Check which diagnostic columns exist

existing_diagnostics = [
    col for col in diagnostic_columns
    if col in training_engineered.columns
]

print("Diagnostic columns found:")
print(existing_diagnostics)

print("\nTotal diagnostic columns:",
      len(existing_diagnostics))

Diagnostic columns found:
['calculated_co2_component', 'calculated_co2', 'calculated_cost', 'cost_base', 'cost_adjustment', 'cost_multiplier', 'cost_formula_difference', 'co2_formula_difference', 'cost_relative_error_percent']

Total diagnostic columns: 9


In [35]:
# Remove temporary diagnostic columns
training_model_data = training_engineered.drop(
    columns=existing_diagnostics
)

print("Original integrated shape:",
      training_engineered.shape)

print("Clean integrated shape:",
      training_model_data.shape)

print("\nRemaining diagnostic columns:",
      [
          col for col in diagnostic_columns
          if col in training_model_data.columns
      ]
      )

Original integrated shape: (36000, 34)
Clean integrated shape: (36000, 25)

Remaining diagnostic columns: []


In [36]:
print("FINAL CLEAN DATASET COLUMNS")

for i, column in enumerate(
    training_model_data.columns, start=1
):
    print(f"{i}. {column}")

print("\nDataset shape:", training_model_data.shape)

FINAL CLEAN DATASET COLUMNS
1. product_id
2. material_id
3. industry
4. product_weight_kg
5. fragility_score
6. moisture_sensitive
7. required_strength_score
8. budget_per_pack_inr
9. material_category
10. strength_score
11. max_weight_capacity_kg
12. biodegradability_score
13. recyclability_percent
14. co2_kg_per_kg_material
15. cost_inr_per_kg
16. moisture_resistance_score
17. durability_score
18. packaging_mass_kg_synthetic
19. cost_target_inr_per_pack
20. co2_target_kgco2e_per_pack
21. suitability_score
22. suitable_label
23. co2_impact_index
24. cost_efficiency_index
25. material_suitability_score

Dataset shape: (36000, 25)


In [37]:
# Investigate packaging mass

training_model_data["mass_ratio"] = (
    training_model_data["packaging_mass_kg_synthetic"]
    / training_model_data["product_weight_kg"]
)

print("Packaging mass statistics:")
display(
    training_model_data[
        [
            "product_weight_kg",
            "packaging_mass_kg_synthetic",
            "mass_ratio"
        ]
    ].describe().round(4)
)

print("\nCorrelation with product weight:")

display(
    training_model_data[
        [
            "product_weight_kg",
            "packaging_mass_kg_synthetic"
        ]
    ].corr().round(4)
)

Packaging mass statistics:


,product_weight_kg,packaging_mass_kg_synthetic,mass_ratio
count,36000.0000,36000.0000,36000.0000
mean,4.0743,0.3059,0.0769
std,2.3402,0.1749,0.0117
min,0.1300,0.0250,0.0749
25%,1.9175,0.1438,0.0750
50%,4.1500,0.3113,0.0750
75%,6.1100,0.4583,0.0750
max,7.9800,0.5985,0.1923



Correlation with product weight:


,product_weight_kg,packaging_mass_kg_synthetic
product_weight_kg,1.0000,0.9999
packaging_mass_kg_synthetic,0.9999,1.0000


In [38]:
# Find records where mass ratio differs from 7.5%

mass_exceptions = training_model_data[
    ~training_model_data["mass_ratio"].between(
        0.074, 0.076
    )
]

print("Total exceptional records:", len(mass_exceptions))

display(
    mass_exceptions[
        [
            "product_id",
            "material_id",
            "product_weight_kg",
            "packaging_mass_kg_synthetic",
            "mass_ratio"
        ]
    ].head(15).round(4)
)

Total exceptional records: 1320


,product_id,material_id,product_weight_kg,packaging_mass_kg_synthetic,mass_ratio
840,P0008,M0101,0.26,0.025,0.0962
841,P0008,M0102,0.26,0.025,0.0962
842,P0008,M0103,0.26,0.025,0.0962
843,P0008,M0104,0.26,0.025,0.0962
844,P0008,M0105,0.26,0.025,0.0962
845,P0008,M0106,0.26,0.025,0.0962
846,P0008,M0107,0.26,0.025,0.0962
847,P0008,M0108,0.26,0.025,0.0962
848,P0008,M0109,0.26,0.025,0.0962
849,P0008,M0110,0.26,0.025,0.0962


In [39]:
# Summarize exceptional records by product

exception_summary = (
    mass_exceptions.groupby("product_id")
    .agg(
        product_weight=("product_weight_kg", "first"),
        packaging_mass=("packaging_mass_kg_synthetic", "first"),
        mass_ratio=("mass_ratio", "first"),
        material_count=("material_id", "nunique")
    )
    .sort_values("product_weight")
)

display(exception_summary.round(4))

print("\nNumber of exceptional products:",
      len(exception_summary))

,product_weight,packaging_mass,mass_ratio,material_count
product_id,,,,
P0245,0.13,0.025,0.1923,120
P0183,0.15,0.025,0.1667,120
P0278,0.16,0.025,0.1562,120
P0091,0.18,0.025,0.1389,120
P0228,0.19,0.025,0.1316,120
P0010,0.21,0.025,0.1190,120
P0248,0.22,0.025,0.1136,120
P0082,0.23,0.025,0.1087,120
P0175,0.24,0.025,0.1042,120



Number of exceptional products: 11


In [40]:
# Proposed packaging mass formula

training_model_data["estimated_packaging_mass"] = (
    training_model_data["product_weight_kg"] * 0.075
).clip(lower=0.025)

# Compare estimated and synthetic packaging mass

training_model_data["mass_difference"] = (
    training_model_data["packaging_mass_kg_synthetic"]
    - training_model_data["estimated_packaging_mass"]
).abs()

print("Maximum difference:",
      training_model_data["mass_difference"].max())

print("Mean difference:",
      training_model_data["mass_difference"].mean())

print("Records matching within 0.001 kg:",
      (
          training_model_data["mass_difference"] <= 0.001
      ).sum(),
      "/",
      len(training_model_data)
)

Maximum difference: 5.000000000002225e-05
Mean difference: 2.233333333332494e-05
Records matching within 0.001 kg: 36000 / 36000


In [41]:
# Calculate expected cost and CO2 using estimated mass

training_model_data["expected_cost"] = (
    training_model_data["estimated_packaging_mass"]
    * training_model_data["cost_inr_per_kg"]
    * 1.12
)

training_model_data["expected_co2"] = (
    training_model_data["estimated_packaging_mass"]
    * training_model_data["co2_kg_per_kg_material"]
)

print("Expected cost mean:")
print(training_model_data["expected_cost"].mean().round(4))

print("\nExpected CO2 mean:")
print(training_model_data["expected_co2"].mean().round(4))

print("\nCost target mean:")
print(
    training_model_data["cost_target_inr_per_pack"]
    .mean().round(4)
)

print("\nCO2 target mean:")
print(
    training_model_data["co2_target_kgco2e_per_pack"]
    .mean().round(4)
)

Expected cost mean:
18.8166

Expected CO2 mean:
0.626

Cost target mean:
18.8166

CO2 target mean:
0.626


In [42]:
# Final common input features for regression models

model_features = [
    "industry",
    "product_weight_kg",
    "fragility_score",
    "moisture_sensitive",
    "required_strength_score",
    "budget_per_pack_inr",
    "material_category",
    "strength_score",
    "max_weight_capacity_kg",
    "biodegradability_score",
    "recyclability_percent",
    "co2_kg_per_kg_material",
    "cost_inr_per_kg",
    "moisture_resistance_score",
    "durability_score",
    "estimated_packaging_mass"
]

cost_target = "cost_target_inr_per_pack"
co2_target = "co2_target_kgco2e_per_pack"

X = training_model_data[model_features]

y_cost = training_model_data[cost_target]
y_co2 = training_model_data[co2_target]

print("Input features:", X.shape)
print("Cost target:", y_cost.shape)
print("CO2 target:", y_co2.shape)

display(X.head())

Input features: (36000, 16)
Cost target: (36000,)
CO2 target: (36000,)


,industry,product_weight_kg,fragility_score,moisture_sensitive,required_strength_score,budget_per_pack_inr,material_category,strength_score,max_weight_capacity_kg,biodegradability_score,recyclability_percent,co2_kg_per_kg_material,cost_inr_per_kg,moisture_resistance_score,durability_score,estimated_packaging_mass
0,Food,2.12,39,1,46,12.44,Paper,58.1,5.09,94,86,0.834,31.66,26,37,0.159
1,Food,2.12,39,1,46,12.44,Paper,50.9,4.99,86,84,0.739,27.19,27,36,0.159
2,Food,2.12,39,1,46,12.44,Paper,51.5,4.41,97,91,0.800,29.46,33,42,0.159
3,Food,2.12,39,1,46,12.44,Paper,55.5,4.09,94,95,0.797,24.77,31,38,0.159
4,Food,2.12,39,1,46,12.44,Paper,48.3,3.96,92,86,0.819,25.85,26,45,0.159


In [43]:
from sklearn.model_selection import train_test_split

# Split input features and both targets together
X_train, X_test, y_cost_train, y_cost_test, y_co2_train, y_co2_test = (
    train_test_split(
        X,
        y_cost,
        y_co2,
        test_size=0.20,
        random_state=42
    )
)

print("Training input:", X_train.shape)
print("Testing input:", X_test.shape)

print("Cost training target:", y_cost_train.shape)
print("Cost testing target:", y_cost_test.shape)

print("CO2 training target:", y_co2_train.shape)
print("CO2 testing target:", y_co2_test.shape)

Training input: (28800, 16)
Testing input: (7200, 16)
Cost training target: (28800,)
Cost testing target: (7200,)
CO2 training target: (28800,)
CO2 testing target: (7200,)


In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Identify categorical and numerical features
categorical_features = [
    "industry",
    "material_category"
]

numerical_features = [
    col for col in model_features
    if col not in categorical_features
    and col != "moisture_sensitive"
]

binary_features = ["moisture_sensitive"]

# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "binary",
            "passthrough",
            binary_features
        )
    ]
)

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform testing data without refitting
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (28800, 30)
Processed testing shape: (7200, 30)


In [45]:
import joblib
from pathlib import Path

# Create models directory if it doesn't exist
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save fitted preprocessing pipeline
preprocessor_path = MODEL_DIR / "preprocessor.pkl"

joblib.dump(preprocessor, preprocessor_path)

print("Preprocessor saved successfully!")
print("Location:", preprocessor_path)

Preprocessor saved successfully!
Location: c:\Users\gouth\OneDrive\Pictures\Documents\EcoPackAI_Project\models\preprocessor.pkl


In [46]:
loaded_preprocessor = joblib.load(preprocessor_path)

print("Preprocessor loaded successfully!")

print(
    "Training shape:",
    loaded_preprocessor.transform(X_train).shape
)

print(
    "Testing shape:",
    loaded_preprocessor.transform(X_test).shape
)

Preprocessor loaded successfully!
Training shape: (28800, 30)
Testing shape: (7200, 30)


In [47]:
import joblib

# Save prepared datasets
joblib.dump(X_train_processed, MODEL_DIR / "X_train.pkl")
joblib.dump(X_test_processed, MODEL_DIR / "X_test.pkl")

joblib.dump(y_cost_train, MODEL_DIR / "y_cost_train.pkl")
joblib.dump(y_cost_test, MODEL_DIR / "y_cost_test.pkl")

joblib.dump(y_co2_train, MODEL_DIR / "y_co2_train.pkl")
joblib.dump(y_co2_test, MODEL_DIR / "y_co2_test.pkl")

print("All prepared datasets saved successfully!")

All prepared datasets saved successfully!


In [48]:
saved_files = [
    "preprocessor.pkl",
    "X_train.pkl",
    "X_test.pkl",
    "y_cost_train.pkl",
    "y_cost_test.pkl",
    "y_co2_train.pkl",
    "y_co2_test.pkl"
]

for filename in saved_files:
    file_path = MODEL_DIR / filename
    print(filename, ":", file_path.exists())

preprocessor.pkl : True
X_train.pkl : True
X_test.pkl : True
y_cost_train.pkl : True
y_cost_test.pkl : True
y_co2_train.pkl : True
y_co2_test.pkl : True


In [49]:
# MODULE 4: PRE-TRAINING VALIDATION

import numpy as np

print("========== DATA VALIDATION ==========")

# Check feature dimensions
print("X_train shape:", X_train_processed.shape)
print("X_test shape:", X_test_processed.shape)

# Check target dimensions
print("Cost training:", len(y_cost_train))
print("Cost testing:", len(y_cost_test))
print("CO2 training:", len(y_co2_train))
print("CO2 testing:", len(y_co2_test))

# Check invalid values
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()

assert np.isfinite(y_cost_train).all()
assert np.isfinite(y_cost_test).all()
assert np.isfinite(y_co2_train).all()
assert np.isfinite(y_co2_test).all()

# Check alignment
assert X_train_processed.shape[0] == len(y_cost_train)
assert X_test_processed.shape[0] == len(y_cost_test)

assert X_train_processed.shape[0] == len(y_co2_train)
assert X_test_processed.shape[0] == len(y_co2_test)

# Check target values
assert (y_cost_train >= 0).all()
assert (y_cost_test >= 0).all()
assert (y_co2_train >= 0).all()
assert (y_co2_test >= 0).all()

# Check feature count
assert X_train_processed.shape[1] == 30

print("\nAll validation checks passed!")
print("Ready for model training.")

========== DATA VALIDATION ==========
X_train shape: (28800, 30)
X_test shape: (7200, 30)
Cost training: 28800
Cost testing: 7200
CO2 training: 28800
CO2 testing: 7200

All validation checks passed!
Ready for model training.


In [50]:
from sklearn.ensemble import RandomForestRegressor
import joblib

# Create Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train_processed, y_cost_train)

print("Random Forest model trained successfully!")

# Predict packaging cost
cost_predictions = rf_model.predict(X_test_processed)

print("Training records:", len(y_cost_train))
print("Testing predictions:", len(cost_predictions))

# Compare actual and predicted costs
cost_results = pd.DataFrame({
    "Actual Cost (INR)": y_cost_test.values[:10],
    "Predicted Cost (INR)": cost_predictions[:10]
})

display(cost_results.round(4))

# Save trained model
model_path = MODEL_DIR / "cost_random_forest.pkl"
joblib.dump(rf_model, model_path)

print("Model saved at:", model_path)

Random Forest model trained successfully!
Training records: 28800
Testing predictions: 7200


,Actual Cost (INR),Predicted Cost (INR)
0,2.5957,2.5912
1,9.3530,9.3682
2,7.2071,7.1825
3,24.9883,24.9594
4,26.7214,26.7948
5,14.8508,14.8734
6,10.8538,10.8823
7,13.9823,14.0225
8,6.5703,6.5308
9,26.3340,26.4746


Model saved at: c:\Users\gouth\OneDrive\Pictures\Documents\EcoPackAI_Project\models\cost_random_forest.pkl


In [51]:
# MODULE 4: RANDOM FOREST MODEL EVALUATION

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Calculate evaluation metrics
cost_rmse = np.sqrt(
    mean_squared_error(y_cost_test, cost_predictions)
)

cost_mae = mean_absolute_error(
    y_cost_test, cost_predictions
)

cost_r2 = r2_score(
    y_cost_test, cost_predictions
)

# Display results
print("========== RANDOM FOREST EVALUATION ==========")

print(f"RMSE: {cost_rmse:.6f}")
print(f"MAE : {cost_mae:.6f}")
print(f"R2 Score: {cost_r2:.6f}")

========== RANDOM FOREST EVALUATION ==========
RMSE: 0.062252
MAE : 0.038075
R2 Score: 0.999982


In [52]:
# MODULE 4: XGBOOST CO2 PREDICTION

from xgboost import XGBRegressor
import joblib

# 1. Initialize XGBoost model
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# 2. Train the model
xgb_model.fit(
    X_train_processed,
    y_co2_train
)

print("XGBoost CO2 Prediction Model trained successfully!")

# 3. Predict CO2 footprint
co2_predictions = xgb_model.predict(X_test_processed)

print("Training records:", len(y_co2_train))
print("Testing predictions:", len(co2_predictions))

# 4. Compare actual and predicted CO2 values
co2_results = pd.DataFrame({
    "Actual CO2 (kg CO2e)": y_co2_test.values[:10],
    "Predicted CO2 (kg CO2e)": co2_predictions[:10]
})

display(co2_results.round(6))

# 5. Save trained model
model_path = MODEL_DIR / "co2_xgboost.pkl"

joblib.dump(xgb_model, model_path)

print("XGBoost model saved at:", model_path)

XGBoost CO2 Prediction Model trained successfully!
Training records: 28800
Testing predictions: 7200


,Actual CO2 (kg CO2e),Predicted CO2 (kg CO2e)
0,0.0654,0.065675
1,0.2208,0.220563
2,0.1695,0.169277
3,0.9009,0.900402
4,0.7584,0.758436
5,0.3474,0.349848
6,0.3081,0.314098
7,0.2314,0.233440
8,0.1151,0.114091
9,0.8037,0.809611


XGBoost model saved at: c:\Users\gouth\OneDrive\Pictures\Documents\EcoPackAI_Project\models\co2_xgboost.pkl


In [53]:
# MODULE 4: XGBOOST MODEL EVALUATION

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Calculate evaluation metrics
co2_rmse = np.sqrt(
    mean_squared_error(y_co2_test, co2_predictions)
)

co2_mae = mean_absolute_error(
    y_co2_test, co2_predictions
)

co2_r2 = r2_score(
    y_co2_test, co2_predictions
)

# Display results
print("========== XGBOOST EVALUATION ==========")

print(f"RMSE: {co2_rmse:.8f}")
print(f"MAE : {co2_mae:.8f}")
print(f"R2 Score: {co2_r2:.8f}")

========== XGBOOST EVALUATION ==========
RMSE: 0.00407390
MAE : 0.00232776
R2 Score: 0.99998104


In [54]:
# VERIFY MODULE 4 MODEL FILES

cost_model_path = MODEL_DIR / "cost_random_forest.pkl"
co2_model_path = MODEL_DIR / "co2_xgboost.pkl"

print("Random Forest model:", cost_model_path.exists())
print("XGBoost model:", co2_model_path.exists())

assert cost_model_path.exists()
assert co2_model_path.exists()

print("\nBoth trained models are available!")

Random Forest model: True
XGBoost model: True

Both trained models are available!


In [55]:
# MODULE 4: RECOMMENDATION DATA INSPECTION

from pathlib import Path
import pandas as pd

# Locate project root
project_root = Path.cwd()

while not (project_root / "data").exists():
    if project_root == project_root.parent:
        raise FileNotFoundError("Project data folder not found.")
    project_root = project_root.parent

# Load engineered materials and cleaned products
materials_df = pd.read_csv(
    project_root / "data" / "processed" / "materials_feature_engineered.csv"
)

products_df = pd.read_csv(
    project_root / "data" / "processed" / "products_cleaned.csv"
)

print("========== MATERIAL DATA ==========")
print("Shape:", materials_df.shape)
print("Columns:")
print(materials_df.columns.tolist())

print("\n========== PRODUCT DATA ==========")
print("Shape:", products_df.shape)
print("Columns:")
print(products_df.columns.tolist())

print("\nMaterial records:", len(materials_df))
print("Product records:", len(products_df))

========== MATERIAL DATA ==========
Shape: (120, 14)
Columns:
['material_id', 'material_name', 'material_category', 'strength_score', 'max_weight_capacity_kg', 'biodegradability_score', 'recyclability_percent', 'co2_kg_per_kg_material', 'cost_inr_per_kg', 'moisture_resistance_score', 'durability_score', 'co2_impact_index', 'cost_efficiency_index', 'material_suitability_score']

========== PRODUCT DATA ==========
Shape: (300, 8)
Columns:
['product_id', 'industry', 'product_type', 'product_weight_kg', 'fragility_score', 'moisture_sensitive', 'required_strength_score', 'budget_per_pack_inr']

Material records: 120
Product records: 300


In [56]:
# MODULE 4: MATERIAL AND PRODUCT REQUIREMENT ANALYSIS

print("========== MATERIAL PROPERTY SUMMARY ==========")

material_columns = [
    "strength_score",
    "max_weight_capacity_kg",
    "biodegradability_score",
    "recyclability_percent",
    "co2_kg_per_kg_material",
    "cost_inr_per_kg",
    "moisture_resistance_score",
    "durability_score",
    "co2_impact_index",
    "cost_efficiency_index",
    "material_suitability_score"
]

display(
    materials_df[material_columns]
    .describe()
    .round(3)
)

print("\n========== PRODUCT REQUIREMENT SUMMARY ==========")

product_columns = [
    "product_weight_kg",
    "fragility_score",
    "moisture_sensitive",
    "required_strength_score",
    "budget_per_pack_inr"
]

display(
    products_df[product_columns]
    .describe()
    .round(3)
)

print("\n========== MOISTURE SENSITIVITY VALUES ==========")
print(products_df["moisture_sensitive"].value_counts(dropna=False))

print("\n========== MATERIAL CATEGORIES ==========")
print(materials_df["material_category"].value_counts())

========== MATERIAL PROPERTY SUMMARY ==========


,strength_score,max_weight_capacity_kg,biodegradability_score,recyclability_percent,co2_kg_per_kg_material,cost_inr_per_kg,moisture_resistance_score,durability_score,co2_impact_index,cost_efficiency_index,material_suitability_score
count,120.000,120.000,120.000,120.000,120.000,120.000,120.000,120.000,120.000,120.000,120.000
mean,74.265,13.454,57.350,75.808,2.046,54.918,58.267,66.300,19.686,46.923,66.398
std,14.243,8.278,39.945,23.004,2.377,24.950,24.673,21.064,22.871,21.968,7.271
min,46.700,3.960,0.000,23.000,0.616,20.720,21.000,31.000,5.926,19.649,53.080
25%,62.600,6.975,15.000,64.000,0.804,31.675,38.000,46.750,7.730,26.416,61.330
50%,74.700,10.925,81.500,85.000,1.080,48.155,51.000,64.500,10.395,43.091,65.300
75%,85.475,17.238,92.250,93.250,2.247,78.438,79.250,83.000,21.618,65.414,71.945
max,100.000,35.150,100.000,100.000,10.394,105.450,100.000,100.000,100.000,100.000,80.200



========== PRODUCT REQUIREMENT SUMMARY ==========


,product_weight_kg,fragility_score,moisture_sensitive,required_strength_score,budget_per_pack_inr
count,300.000,300.000,300.000,300.000,300.000
mean,4.074,50.543,0.443,56.027,10.910
std,2.344,28.838,0.498,16.841,6.209
min,0.130,1.000,0.000,20.000,1.000
25%,1.918,26.000,0.000,44.000,5.538
50%,4.150,51.000,0.000,55.500,10.330
75%,6.110,75.000,1.000,68.250,16.280
max,7.980,100.000,1.000,98.000,21.990



========== MOISTURE SENSITIVITY VALUES ==========
moisture_sensitive
0    167
1    133
Name: count, dtype: int64

========== MATERIAL CATEGORIES ==========
material_category
Paper          40
Plastic        30
Plant Fiber    20
Bioplastic     10
Glass          10
Metal          10
Name: count, dtype: int64


In [57]:
# MODULE 4: EXISTING SUITABILITY ANALYSIS

training_df = pd.read_csv(
    project_root / "data" / "processed" /
    "training_data_feature_engineered.csv"
)

print("========== SUITABILITY LABEL DISTRIBUTION ==========")

print(training_df["suitable_label"].value_counts())

print("\n========== SUITABILITY SCORE SUMMARY ==========")

print(
    training_df["suitability_score"]
    .describe()
    .round(3)
)

print("\n========== SUITABILITY BY MATERIAL CATEGORY ==========")

display(
    training_df.groupby("material_category")
    .agg(
        total_records=("suitable_label", "count"),
        suitable_records=("suitable_label", "sum"),
        average_suitability=("suitability_score", "mean")
    )
    .round(3)
)

print("\n========== SUITABILITY BY MATERIAL ==========")

display(
    training_df.groupby("material_id")
    .agg(
        suitable_rate=("suitable_label", "mean"),
        average_score=("suitability_score", "mean")
    )
    .describe()
    .round(3)
)

========== SUITABILITY LABEL DISTRIBUTION ==========
suitable_label
0    29466
1     6534
Name: count, dtype: int64

========== SUITABILITY SCORE SUMMARY ==========
count    36000.000
mean        80.994
std          7.175
min         54.830
25%         75.790
50%         80.840
75%         86.260
max         95.650
Name: suitability_score, dtype: float64

========== SUITABILITY BY MATERIAL CATEGORY ==========


,total_records,suitable_records,average_suitability
material_category,,,
Bioplastic,3000,491,76.023
Glass,3000,562,79.251
Metal,3000,529,79.058
Paper,12000,1829,84.750
Plant Fiber,6000,1090,87.678
Plastic,9000,2033,74.411



========== SUITABILITY BY MATERIAL ==========


,suitable_rate,average_score
count,120.000,120.000
mean,0.182,80.994
std,0.056,5.529
min,0.063,70.006
25%,0.153,76.716
50%,0.180,79.850
75%,0.203,86.171
max,0.360,89.950


In [58]:
# MODULE 4: SUITABILITY RULE INVESTIGATION

print("========== SUITABILITY SCORE BY LABEL ==========")

display(
    training_df.groupby("suitable_label")
    .agg(
        total_records=("product_id", "count"),
        average_score=("suitability_score", "mean"),
        average_strength=("strength_score", "mean"),
        average_required_strength=("required_strength_score", "mean"),
        average_capacity=("max_weight_capacity_kg", "mean"),
        average_product_weight=("product_weight_kg", "mean"),
        average_budget=("budget_per_pack_inr", "mean"),
        average_cost=("cost_target_inr_per_pack", "mean")
    )
    .round(3)
)

print("\n========== SUITABILITY BY MOISTURE SENSITIVITY ==========")

display(
    training_df.groupby(
        ["moisture_sensitive", "suitable_label"]
    ).size().unstack(fill_value=0)
)

print("\n========== SUITABILITY SCORE DISTRIBUTION BY LABEL ==========")

display(
    training_df.groupby("suitable_label")["suitability_score"]
    .describe()
    .round(3)
)

========== SUITABILITY SCORE BY LABEL ==========


,total_records,average_score,average_strength,average_required_strength,average_capacity,average_product_weight,average_budget,average_cost
suitable_label,,,,,,,,
0,29466,79.767,73.759,57.260,13.297,4.607,10.092,21.468
1,6534,86.526,76.549,50.467,14.160,1.672,14.599,6.859



========== SUITABILITY BY MOISTURE SENSITIVITY ==========


suitable_label,0,1
moisture_sensitive,,
0,15554,4486
1,13912,2048



========== SUITABILITY SCORE DISTRIBUTION BY LABEL ==========


,count,mean,std,min,25%,50%,75%,max
suitable_label,,,,,,,,
0,29466.0,79.767,6.784,54.83,74.98,79.45,85.45,95.62
1,6534.0,86.526,6.218,74.00,81.05,85.35,92.70,95.65


In [59]:
# MODULE 4: TECHNICAL ELIGIBILITY ANALYSIS

# Combine all product and material combinations
candidate_analysis = products_df.merge(
    materials_df,
    how="cross"
)

# Check basic technical requirements
candidate_analysis["strength_pass"] = (
    candidate_analysis["strength_score"]
    >= candidate_analysis["required_strength_score"]
)

candidate_analysis["capacity_pass"] = (
    candidate_analysis["max_weight_capacity_kg"]
    >= candidate_analysis["product_weight_kg"]
)

# Both requirements must pass
candidate_analysis["technical_pass"] = (
    candidate_analysis["strength_pass"]
    & candidate_analysis["capacity_pass"]
)

print("Total product-material combinations:",
      len(candidate_analysis))

print("\nStrength requirement passed:",
      candidate_analysis["strength_pass"].sum())

print("Weight capacity passed:",
      candidate_analysis["capacity_pass"].sum())

print("Both technical requirements passed:",
      candidate_analysis["technical_pass"].sum())

print("\nTechnical eligibility percentage:",
      round(candidate_analysis["technical_pass"].mean() * 100, 2), "%")

Total product-material combinations: 36000

Strength requirement passed: 28427
Weight capacity passed: 32638
Both technical requirements passed: 26686

Technical eligibility percentage: 74.13 %


In [61]:
# MODULE 4: COST AND BUDGET ELIGIBILITY ANALYSIS
# CORRECTED VERSION

import joblib

# 1. Load the trained Random Forest model
cost_model = joblib.load(
    MODEL_DIR / "cost_random_forest.pkl"
)

# 2. Create model input from candidate combinations
recommendation_features = candidate_analysis.copy()

# 3. Recreate packaging mass using the training formula
recommendation_features["estimated_packaging_mass"] = (
    0.075 * recommendation_features["product_weight_kg"]
).clip(lower=0.025)

# 4. Verify required features are available
missing_features = [
    col for col in model_features
    if col not in recommendation_features.columns
]

assert not missing_features, (
    f"Missing model features: {missing_features}"
)

# 5. Select features in the exact training order
recommendation_features = recommendation_features[
    model_features
].copy()

# 6. Apply the saved preprocessing pipeline
recommendation_features_processed = preprocessor.transform(
    recommendation_features
)

# 7. Predict packaging cost
candidate_analysis["predicted_cost_inr"] = (
    cost_model.predict(recommendation_features_processed)
)

# 8. Check budget eligibility
candidate_analysis["budget_pass"] = (
    candidate_analysis["predicted_cost_inr"]
    <= candidate_analysis["budget_per_pack_inr"]
)

# 9. Combine technical and budget eligibility
candidate_analysis["technical_budget_pass"] = (
    candidate_analysis["technical_pass"]
    & candidate_analysis["budget_pass"]
)

# 10. Display results
print("========== COST ELIGIBILITY RESULTS ==========")

print("Total combinations:", len(candidate_analysis))

print(
    "Within budget:",
    candidate_analysis["budget_pass"].sum()
)

print(
    "Technically eligible and within budget:",
    candidate_analysis["technical_budget_pass"].sum()
)

print(
    "Technical + budget eligibility percentage:",
    round(
        candidate_analysis["technical_budget_pass"].mean() * 100,
        2
    ),
    "%"
)

print("\nCost prediction completed successfully!")

========== COST ELIGIBILITY RESULTS ==========
Total combinations: 36000
Within budget: 12982
Technically eligible and within budget: 9231
Technical + budget eligibility percentage: 25.64 %

Cost prediction completed successfully!


In [62]:
# MODULE 4: CO2 PREDICTION FOR MATERIAL RECOMMENDATIONS

import joblib

# 1. Load trained XGBoost model
co2_model = joblib.load(
    MODEL_DIR / "co2_xgboost.pkl"
)

# 2. Predict CO2 using the same preprocessed features
co2_predictions_all = co2_model.predict(
    recommendation_features_processed
)

# 3. Store predictions
candidate_analysis["predicted_co2_kg"] = (
    co2_predictions_all
)

# 4. Check predictions
print("========== CO2 PREDICTION RESULTS ==========")

print("Total predictions:", len(co2_predictions_all))

print(
    "Minimum predicted CO2:",
    round(co2_predictions_all.min(), 6)
)

print(
    "Maximum predicted CO2:",
    round(co2_predictions_all.max(), 6)
)

print(
    "Average predicted CO2:",
    round(co2_predictions_all.mean(), 6)
)

# 5. Verify predictions
assert len(co2_predictions_all) == 36000
assert np.isfinite(co2_predictions_all).all()
assert (co2_predictions_all >= 0).all()

print("\nCO2 predictions completed successfully!")

========== CO2 PREDICTION RESULTS ==========
Total predictions: 36000
Minimum predicted CO2: 0.014496
Maximum predicted CO2: 6.209336
Average predicted CO2: 0.625985

CO2 predictions completed successfully!


In [63]:
# MODULE 4: ENVIRONMENTAL SCORE INSPECTION

print("========== ENVIRONMENTAL SCORE SUMMARY ==========")

score_columns = [
    "co2_impact_index",
    "cost_efficiency_index",
    "biodegradability_score",
    "recyclability_percent",
    "material_suitability_score"
]

display(
    materials_df[score_columns]
    .describe()
    .round(3)
)

print("\n========== SAMPLE MATERIAL SCORES ==========")

display(
    materials_df[
        [
            "material_name",
            "material_category",
            "co2_impact_index",
            "cost_efficiency_index",
            "biodegradability_score",
            "recyclability_percent"
        ]
    ].head(10).round(3)
)

========== ENVIRONMENTAL SCORE SUMMARY ==========


,co2_impact_index,cost_efficiency_index,biodegradability_score,recyclability_percent,material_suitability_score
count,120.000,120.000,120.000,120.000,120.000
mean,19.686,46.923,57.350,75.808,66.398
std,22.871,21.968,39.945,23.004,7.271
min,5.926,19.649,0.000,23.000,53.080
25%,7.730,26.416,15.000,64.000,61.330
50%,10.395,43.091,81.500,85.000,65.300
75%,21.618,65.414,92.250,93.250,71.945
max,100.000,100.000,100.000,100.000,80.200



========== SAMPLE MATERIAL SCORES ==========


,material_name,material_category,co2_impact_index,cost_efficiency_index,biodegradability_score,recyclability_percent
0,Kraft Paper,Paper,8.024,65.445,94,86
1,Kraft Paper - Variant 02,Paper,7.110,76.204,86,84
2,Kraft Paper - Variant 03,Paper,7.697,70.333,97,91
3,Kraft Paper - Variant 04,Paper,7.668,83.650,94,95
4,Kraft Paper - Variant 05,Paper,7.880,80.155,92,86
5,Kraft Paper - Variant 06,Paper,6.369,79.877,99,83
6,Kraft Paper - Variant 07,Paper,7.697,65.924,98,84
7,Kraft Paper - Variant 08,Paper,7.254,81.128,89,89
8,Kraft Paper - Variant 09,Paper,7.081,71.473,91,81
9,Kraft Paper - Variant 10,Paper,7.341,84.814,98,95


In [64]:
# MODULE 4: RECOMMENDATION CONFIGURATION

MIN_MOISTURE_RESISTANCE = 60

print("========== RECOMMENDATION RULES ==========")

print("Minimum moisture resistance:",
      MIN_MOISTURE_RESISTANCE)

print("Strength requirement: Material strength >= Product required strength")
print("Weight requirement: Material capacity >= Product weight")
print("Budget requirement: Predicted cost <= Product budget")
print("Moisture requirement: Applied only to moisture-sensitive products")

print("\nRecommendation configuration established!")

========== RECOMMENDATION RULES ==========
Minimum moisture resistance: 60
Strength requirement: Material strength >= Product required strength
Weight requirement: Material capacity >= Product weight
Budget requirement: Predicted cost <= Product budget
Moisture requirement: Applied only to moisture-sensitive products

Recommendation configuration established!


In [65]:
# MODULE 4: ENVIRONMENTAL RANKING CONFIGURATION

RANKING_WEIGHTS = {
    "co2": 0.40,
    "biodegradability": 0.25,
    "recyclability": 0.20,
    "cost": 0.15
}

# Verify total weight
assert abs(sum(RANKING_WEIGHTS.values()) - 1.0) < 1e-9

print("========== RANKING CONFIGURATION ==========")

for factor, weight in RANKING_WEIGHTS.items():
    print(f"{factor}: {weight * 100:.0f}%")

print("\nTotal weight:", sum(RANKING_WEIGHTS.values()) * 100, "%")
print("Ranking configuration validated!")

========== RANKING CONFIGURATION ==========
co2: 40%
biodegradability: 25%
recyclability: 20%
cost: 15%

Total weight: 100.0 %
Ranking configuration validated!


In [66]:
# MODULE 4: ECOPACKAI MATERIAL RECOMMENDATION ENGINE

def recommend_materials(product_id, top_n=10):

    # -----------------------------------------
    # 1. Validate input
    # -----------------------------------------

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    product_matches = products_df[
        products_df["product_id"] == product_id
    ]

    if product_matches.empty:
        raise ValueError(f"Product ID {product_id} not found.")

    if len(product_matches) > 1:
        raise ValueError("Product ID must be unique.")

    # -----------------------------------------
    # 2. Get candidate materials for product
    # -----------------------------------------

    candidates = candidate_analysis[
        candidate_analysis["product_id"] == product_id
    ].copy()

    if candidates.empty:
        raise ValueError("No candidate materials found.")

    # -----------------------------------------
    # 3. Apply technical eligibility
    # -----------------------------------------

    candidates["technical_pass"] = (
        (candidates["strength_score"] >=
         candidates["required_strength_score"])
        &
        (candidates["max_weight_capacity_kg"] >=
         candidates["product_weight_kg"])
    )

    # -----------------------------------------
    # 4. Apply moisture requirement
    # -----------------------------------------

    candidates["moisture_pass"] = (
        (candidates["moisture_sensitive"] == 0)
        |
        (candidates["moisture_resistance_score"] >=
         MIN_MOISTURE_RESISTANCE)
    )

    # -----------------------------------------
    # 5. Apply budget requirement
    # -----------------------------------------

    candidates["budget_pass"] = (
        candidates["predicted_cost_inr"]
        <= candidates["budget_per_pack_inr"]
    )

    # -----------------------------------------
    # 6. Keep only eligible materials
    # -----------------------------------------

    eligible = candidates[
        candidates["technical_pass"]
        & candidates["moisture_pass"]
        & candidates["budget_pass"]
    ].copy()

    if eligible.empty:
        return eligible

    # -----------------------------------------
    # 7. Normalize prediction scores
    # Lower cost and CO2 = higher score
    # -----------------------------------------

    def lower_is_better(series):

        minimum = series.min()
        maximum = series.max()

        if maximum == minimum:
            return pd.Series(100.0, index=series.index)

        return (
            (maximum - series) /
            (maximum - minimum) * 100
        )

    eligible["co2_score"] = lower_is_better(
        eligible["predicted_co2_kg"]
    )

    eligible["cost_score"] = lower_is_better(
        eligible["predicted_cost_inr"]
    )

    # -----------------------------------------
    # 8. Environmental and economic scoring
    # -----------------------------------------

    eligible["final_score"] = (
        eligible["co2_score"] *
        RANKING_WEIGHTS["co2"]
        +
        eligible["biodegradability_score"] *
        RANKING_WEIGHTS["biodegradability"]
        +
        eligible["recyclability_percent"] *
        RANKING_WEIGHTS["recyclability"]
        +
        eligible["cost_score"] *
        RANKING_WEIGHTS["cost"]
    )

    # -----------------------------------------
    # 9. Generate recommendation explanations
    # -----------------------------------------

    eligible["recommendation_reason"] = (
        "Passed strength, weight capacity, moisture and budget checks."
    )

    # -----------------------------------------
    # 10. Rank materials
    # -----------------------------------------

    eligible = eligible.sort_values(
        by=["final_score", "predicted_co2_kg", "predicted_cost_inr"],
        ascending=[False, True, True],
        kind="stable"
    ).reset_index(drop=True)

    eligible["rank"] = range(1, len(eligible) + 1)

    # -----------------------------------------
    # 11. Return top recommendations
    # -----------------------------------------

    result_columns = [
        "rank",
        "material_id",
        "material_name",
        "material_category",
        "predicted_cost_inr",
        "predicted_co2_kg",
        "biodegradability_score",
        "recyclability_percent",
        "co2_score",
        "cost_score",
        "final_score",
        "recommendation_reason"
    ]

    return eligible[result_columns].head(top_n)


print("EcoPackAI recommendation function created successfully!")

EcoPackAI recommendation function created successfully!


In [67]:
# MODULE 4: FIRST PRODUCT RECOMMENDATION TEST

# Select the first product from the dataset
test_product = products_df.iloc[0]

test_product_id = test_product["product_id"]

print("========== SELECTED PRODUCT ==========")

print("Product ID:", test_product_id)
print("Industry:", test_product["industry"])
print("Product Type:", test_product["product_type"])
print("Product Weight:", test_product["product_weight_kg"], "kg")
print("Fragility Score:", test_product["fragility_score"])
print("Moisture Sensitive:", test_product["moisture_sensitive"])
print("Required Strength:", test_product["required_strength_score"])
print("Budget:", test_product["budget_per_pack_inr"], "INR")

# Generate recommendations
recommendations = recommend_materials(
    product_id=test_product_id,
    top_n=10
)

print("\n========== TOP MATERIAL RECOMMENDATIONS ==========")

if recommendations.empty:
    print("No materials satisfy all eligibility requirements.")
else:
    display(recommendations.round(4))

    print("\nNumber of recommendations returned:",
          len(recommendations))

========== SELECTED PRODUCT ==========
Product ID: P0001
Industry: Food
Product Type: Snack box
Product Weight: 2.12 kg
Fragility Score: 39
Moisture Sensitive: 1
Required Strength: 46
Budget: 12.44 INR

========== TOP MATERIAL RECOMMENDATIONS ==========


,rank,material_id,material_name,material_category,predicted_cost_inr,predicted_co2_kg,biodegradability_score,recyclability_percent,co2_score,cost_score,final_score,recommendation_reason
0,1,M0708,PLA Bioplastic - Variant 08,Bioplastic,11.7877,0.2803,80,54,100.000000,20.4159,73.8624,"Passed strength, weight capacity, moisture and..."
1,2,M0705,PLA Bioplastic - Variant 05,Bioplastic,9.8904,0.3273,80,50,69.403099,100.0000,72.7613,"Passed strength, weight capacity, moisture and..."
2,3,M0807,Recycled PET - Variant 07,Plastic,10.3016,0.3309,28,77,67.097702,82.7497,61.6515,"Passed strength, weight capacity, moisture and..."
3,4,M0809,Recycled PET - Variant 09,Plastic,10.0033,0.3333,25,71,65.514099,95.2612,60.9448,"Passed strength, weight capacity, moisture and..."
4,5,M0806,Recycled PET - Variant 06,Plastic,9.9581,0.3585,29,71,49.096298,97.1569,55.6621,"Passed strength, weight capacity, moisture and..."
5,6,M0805,Recycled PET - Variant 05,Plastic,10.2968,0.3647,28,67,45.058800,82.9522,50.8663,"Passed strength, weight capacity, moisture and..."
6,7,M0901,HDPE,Plastic,11.7105,0.3353,22,39,64.187698,23.6505,42.5227,"Passed strength, weight capacity, moisture and..."
7,8,M0804,Recycled PET - Variant 04,Plastic,10.1569,0.3972,20,68,23.850901,88.8214,41.4636,"Passed strength, weight capacity, moisture and..."
8,9,M0808,Recycled PET - Variant 08,Plastic,12.1279,0.3601,23,73,48.041100,6.1436,40.4880,"Passed strength, weight capacity, moisture and..."
9,10,M0810,Recycled PET - Variant 10,Plastic,11.9954,0.3626,21,72,46.445702,11.7019,39.9836,"Passed strength, weight capacity, moisture and..."



Number of recommendations returned: 10


In [68]:
# DISPLAY TOP 10 RECOMMENDATIONS

display(
    recommendations.round(4)
)

,rank,material_id,material_name,material_category,predicted_cost_inr,predicted_co2_kg,biodegradability_score,recyclability_percent,co2_score,cost_score,final_score,recommendation_reason
0,1,M0708,PLA Bioplastic - Variant 08,Bioplastic,11.7877,0.2803,80,54,100.000000,20.4159,73.8624,"Passed strength, weight capacity, moisture and..."
1,2,M0705,PLA Bioplastic - Variant 05,Bioplastic,9.8904,0.3273,80,50,69.403099,100.0000,72.7613,"Passed strength, weight capacity, moisture and..."
2,3,M0807,Recycled PET - Variant 07,Plastic,10.3016,0.3309,28,77,67.097702,82.7497,61.6515,"Passed strength, weight capacity, moisture and..."
3,4,M0809,Recycled PET - Variant 09,Plastic,10.0033,0.3333,25,71,65.514099,95.2612,60.9448,"Passed strength, weight capacity, moisture and..."
4,5,M0806,Recycled PET - Variant 06,Plastic,9.9581,0.3585,29,71,49.096298,97.1569,55.6621,"Passed strength, weight capacity, moisture and..."
5,6,M0805,Recycled PET - Variant 05,Plastic,10.2968,0.3647,28,67,45.058800,82.9522,50.8663,"Passed strength, weight capacity, moisture and..."
6,7,M0901,HDPE,Plastic,11.7105,0.3353,22,39,64.187698,23.6505,42.5227,"Passed strength, weight capacity, moisture and..."
7,8,M0804,Recycled PET - Variant 04,Plastic,10.1569,0.3972,20,68,23.850901,88.8214,41.4636,"Passed strength, weight capacity, moisture and..."
8,9,M0808,Recycled PET - Variant 08,Plastic,12.1279,0.3601,23,73,48.041100,6.1436,40.4880,"Passed strength, weight capacity, moisture and..."
9,10,M0810,Recycled PET - Variant 10,Plastic,11.9954,0.3626,21,72,46.445702,11.7019,39.9836,"Passed strength, weight capacity, moisture and..."


In [69]:
# MODULE 4: RECOMMENDATION RESULT VALIDATION

# 1. Verify ranking order
assert recommendations["rank"].tolist() == list(
    range(1, len(recommendations) + 1)
)

assert recommendations["final_score"].is_monotonic_decreasing

# 2. Verify budget eligibility
assert (
    recommendations["predicted_cost_inr"]
    <= test_product["budget_per_pack_inr"]
).all()

# 3. Verify non-negative predictions
assert (recommendations["predicted_cost_inr"] >= 0).all()
assert (recommendations["predicted_co2_kg"] >= 0).all()

# 4. Retrieve full candidate details for recommended materials
recommended_details = candidate_analysis[
    candidate_analysis["product_id"] == test_product_id
].merge(
    recommendations[["material_id"]],
    on="material_id",
    how="inner"
)

# 5. Verify technical requirements
assert (
    recommended_details["strength_score"]
    >= recommended_details["required_strength_score"]
).all()

assert (
    recommended_details["max_weight_capacity_kg"]
    >= recommended_details["product_weight_kg"]
).all()

# 6. Verify moisture requirements
assert (
    (
        recommended_details["moisture_sensitive"] == 0
    )
    |
    (
        recommended_details["moisture_resistance_score"]
        >= MIN_MOISTURE_RESISTANCE
    )
).all()

print("========== RECOMMENDATION VALIDATION ==========")
print("Ranking order: PASSED")
print("Budget eligibility: PASSED")
print("Strength requirement: PASSED")
print("Weight capacity: PASSED")
print("Moisture requirement: PASSED")
print("Non-negative predictions: PASSED")

print("\nAll recommendation validation checks passed!")

========== RECOMMENDATION VALIDATION ==========
Ranking order: PASSED
Budget eligibility: PASSED
Strength requirement: PASSED
Weight capacity: PASSED
Moisture requirement: PASSED
Non-negative predictions: PASSED

All recommendation validation checks passed!


In [70]:
# MODULE 4: MOISTURE-SENSITIVE PRODUCT TEST

# Select a moisture-sensitive product
moisture_products = products_df[
    products_df["moisture_sensitive"] == 1
]

assert not moisture_products.empty

moisture_test_product = moisture_products.iloc[0]

moisture_product_id = moisture_test_product["product_id"]

print("========== MOISTURE-SENSITIVE PRODUCT ==========")

print("Product ID:", moisture_product_id)
print("Product Type:", moisture_test_product["product_type"])
print("Product Weight:", moisture_test_product["product_weight_kg"])
print("Required Strength:", moisture_test_product["required_strength_score"])
print("Budget:", moisture_test_product["budget_per_pack_inr"])

# Generate recommendations
moisture_recommendations = recommend_materials(
    product_id=moisture_product_id,
    top_n=10
)

print("\n========== RECOMMENDATION RESULTS ==========")

if moisture_recommendations.empty:
    print("No materials satisfy all requirements.")
else:
    display(moisture_recommendations.round(4))

    print(
        "\nRecommendations returned:",
        len(moisture_recommendations)
    )

========== MOISTURE-SENSITIVE PRODUCT ==========
Product ID: P0001
Product Type: Snack box
Product Weight: 2.12
Required Strength: 46
Budget: 12.44

========== RECOMMENDATION RESULTS ==========


,rank,material_id,material_name,material_category,predicted_cost_inr,predicted_co2_kg,biodegradability_score,recyclability_percent,co2_score,cost_score,final_score,recommendation_reason
0,1,M0708,PLA Bioplastic - Variant 08,Bioplastic,11.7877,0.2803,80,54,100.000000,20.4159,73.8624,"Passed strength, weight capacity, moisture and..."
1,2,M0705,PLA Bioplastic - Variant 05,Bioplastic,9.8904,0.3273,80,50,69.403099,100.0000,72.7613,"Passed strength, weight capacity, moisture and..."
2,3,M0807,Recycled PET - Variant 07,Plastic,10.3016,0.3309,28,77,67.097702,82.7497,61.6515,"Passed strength, weight capacity, moisture and..."
3,4,M0809,Recycled PET - Variant 09,Plastic,10.0033,0.3333,25,71,65.514099,95.2612,60.9448,"Passed strength, weight capacity, moisture and..."
4,5,M0806,Recycled PET - Variant 06,Plastic,9.9581,0.3585,29,71,49.096298,97.1569,55.6621,"Passed strength, weight capacity, moisture and..."
5,6,M0805,Recycled PET - Variant 05,Plastic,10.2968,0.3647,28,67,45.058800,82.9522,50.8663,"Passed strength, weight capacity, moisture and..."
6,7,M0901,HDPE,Plastic,11.7105,0.3353,22,39,64.187698,23.6505,42.5227,"Passed strength, weight capacity, moisture and..."
7,8,M0804,Recycled PET - Variant 04,Plastic,10.1569,0.3972,20,68,23.850901,88.8214,41.4636,"Passed strength, weight capacity, moisture and..."
8,9,M0808,Recycled PET - Variant 08,Plastic,12.1279,0.3601,23,73,48.041100,6.1436,40.4880,"Passed strength, weight capacity, moisture and..."
9,10,M0810,Recycled PET - Variant 10,Plastic,11.9954,0.3626,21,72,46.445702,11.7019,39.9836,"Passed strength, weight capacity, moisture and..."



Recommendations returned: 10


In [71]:
# VERIFY MOISTURE RESISTANCE OF RECOMMENDED MATERIALS

moisture_check = candidate_analysis[
    candidate_analysis["product_id"] == moisture_product_id
].merge(
    moisture_recommendations[["material_id"]],
    on="material_id",
    how="inner"
)

display(
    moisture_check[
        [
            "material_name",
            "moisture_sensitive",
            "moisture_resistance_score",
            "predicted_cost_inr",
            "budget_per_pack_inr"
        ]
    ].round(3)
)

assert (
    moisture_check["moisture_resistance_score"]
    >= MIN_MOISTURE_RESISTANCE
).all()

print("\nAll recommended materials satisfy moisture resistance!")

,material_name,moisture_sensitive,moisture_resistance_score,predicted_cost_inr,budget_per_pack_inr
0,PLA Bioplastic - Variant 05,1,62,9.890,12.44
1,PLA Bioplastic - Variant 08,1,63,11.788,12.44
2,Recycled PET - Variant 04,1,69,10.157,12.44
3,Recycled PET - Variant 05,1,74,10.297,12.44
4,Recycled PET - Variant 06,1,64,9.958,12.44
5,Recycled PET - Variant 07,1,70,10.302,12.44
6,Recycled PET - Variant 08,1,65,12.128,12.44
7,Recycled PET - Variant 09,1,74,10.003,12.44
8,Recycled PET - Variant 10,1,62,11.995,12.44
9,HDPE,1,89,11.711,12.44



All recommended materials satisfy moisture resistance!


In [72]:
# TEST A SECOND MOISTURE-SENSITIVE PRODUCT

other_moisture_products = products_df[
    (products_df["moisture_sensitive"] == 1)
    & (products_df["product_id"] != moisture_product_id)
]

second_moisture_product = other_moisture_products.iloc[0]

second_product_id = second_moisture_product["product_id"]

second_recommendations = recommend_materials(
    product_id=second_product_id,
    top_n=10
)

print("Product ID:", second_product_id)
print("Product Type:", second_moisture_product["product_type"])
print("Product Budget:", second_moisture_product["budget_per_pack_inr"])

print("\nRecommendations returned:", len(second_recommendations))

display(second_recommendations.round(4))

Product ID: P0002
Product Type: Accessory box
Product Budget: 21.6

Recommendations returned: 10


,rank,material_id,material_name,material_category,predicted_cost_inr,predicted_co2_kg,biodegradability_score,recyclability_percent,co2_score,cost_score,final_score,recommendation_reason
0,1,M0708,PLA Bioplastic - Variant 08,Bioplastic,18.1746,0.4398,80,54,62.541801,53.1170,63.7843,"Passed strength, weight capacity, moisture and..."
1,2,M0705,PLA Bioplastic - Variant 05,Bioplastic,15.1701,0.4998,80,50,46.055401,100.0000,63.4222,"Passed strength, weight capacity, moisture and..."
2,3,M1107,Glass - Variant 07,Glass,21.4719,0.3034,0,100,100.000000,1.6629,60.2494,"Passed strength, weight capacity, moisture and..."
3,4,M1110,Glass - Variant 10,Glass,21.4954,0.3169,4,100,96.286102,1.2971,59.7090,"Passed strength, weight capacity, moisture and..."
4,5,M0807,Recycled PET - Variant 07,Plastic,15.8469,0.5078,28,77,43.861900,89.4395,53.3607,"Passed strength, weight capacity, moisture and..."
5,6,M0809,Recycled PET - Variant 09,Plastic,15.3585,0.5087,25,71,43.614601,97.0603,52.4549,"Passed strength, weight capacity, moisture and..."
6,7,M0806,Recycled PET - Variant 06,Plastic,15.2008,0.5432,29,71,34.132702,99.5207,50.0312,"Passed strength, weight capacity, moisture and..."
7,8,M0805,Recycled PET - Variant 05,Plastic,15.9125,0.5614,28,67,29.148199,88.4162,45.3217,"Passed strength, weight capacity, moisture and..."
8,9,M0808,Recycled PET - Variant 08,Plastic,18.7078,0.5463,23,73,33.300400,44.7958,40.3895,"Passed strength, weight capacity, moisture and..."
9,10,M0810,Recycled PET - Variant 10,Plastic,18.4720,0.5552,21,72,30.844299,48.4754,39.2590,"Passed strength, weight capacity, moisture and..."


In [73]:
second_check = candidate_analysis[
    candidate_analysis["product_id"] == second_product_id
].merge(
    second_recommendations[["material_id"]],
    on="material_id",
    how="inner"
)

# Validate all recommendation conditions
assert len(second_check) == len(second_recommendations)
assert (second_check["strength_score"] >= second_check["required_strength_score"]).all()
assert (second_check["max_weight_capacity_kg"] >= second_check["product_weight_kg"]).all()
assert (second_check["moisture_resistance_score"] >= MIN_MOISTURE_RESISTANCE).all()
assert (second_check["predicted_cost_inr"] <= second_check["budget_per_pack_inr"]).all()
assert (second_check["predicted_co2_kg"] >= 0).all()

print("All P0002 recommendation checks passed!")
print("Number of validated recommendations:", len(second_check))

All P0002 recommendation checks passed!
Number of validated recommendations: 10


In [74]:
# Find products with zero eligible materials

eligibility_mask = (
    candidate_analysis["technical_pass"]
    & (
        (candidate_analysis["moisture_sensitive"] == 0)
        | (
            candidate_analysis["moisture_resistance_score"]
            >= MIN_MOISTURE_RESISTANCE
        )
    )
    & candidate_analysis["budget_pass"]
)

eligibility_counts = eligibility_mask.groupby(
    candidate_analysis["product_id"]
).sum()

no_candidate_ids = eligibility_counts[
    eligibility_counts == 0
].index.tolist()

print("Products with no eligible materials:", len(no_candidate_ids))
print("Sample product IDs:", no_candidate_ids[:10])

Products with no eligible materials: 162
Sample product IDs: ['P0003', 'P0007', 'P0011', 'P0012', 'P0013', 'P0015', 'P0016', 'P0019', 'P0020', 'P0021']


In [75]:
no_candidate_product_id = no_candidate_ids[0]

no_recommendations = recommend_materials(
    product_id=no_candidate_product_id,
    top_n=10
)

print("Product ID:", no_candidate_product_id)
print("Recommendations returned:", len(no_recommendations))
print("Returned columns:", no_recommendations.columns.tolist())

display(no_recommendations)

Product ID: P0003
Recommendations returned: 0
Returned columns: ['product_id', 'industry', 'product_type', 'product_weight_kg', 'fragility_score', 'moisture_sensitive', 'required_strength_score', 'budget_per_pack_inr', 'material_id', 'material_name', 'material_category', 'strength_score', 'max_weight_capacity_kg', 'biodegradability_score', 'recyclability_percent', 'co2_kg_per_kg_material', 'cost_inr_per_kg', 'moisture_resistance_score', 'durability_score', 'co2_impact_index', 'cost_efficiency_index', 'material_suitability_score', 'strength_pass', 'capacity_pass', 'technical_pass', 'predicted_cost_inr', 'budget_pass', 'technical_budget_pass', 'predicted_co2_kg', 'moisture_pass']


,product_id,industry,product_type,product_weight_kg,fragility_score,moisture_sensitive,required_strength_score,budget_per_pack_inr,material_id,material_name,...,cost_efficiency_index,material_suitability_score,strength_pass,capacity_pass,technical_pass,predicted_cost_inr,budget_pass,technical_budget_pass,predicted_co2_kg,moisture_pass


In [76]:
# Final recommendation engine status check

print("Recommendation function:", callable(recommend_materials))
print("Materials dataset:", materials_df.shape)
print("Products dataset:", products_df.shape)
print("Candidate combinations:", candidate_analysis.shape[0])
print("Successful test recommendations:", len(second_recommendations))
print("No-result test recommendations:", len(no_recommendations))

print("\nExisting model files:")
print("Cost model:", (MODEL_DIR / "cost_random_forest.pkl").exists())
print("CO2 model:", (MODEL_DIR / "co2_xgboost.pkl").exists())
print("Preprocessor:", (MODEL_DIR / "preprocessor.pkl").exists())

Recommendation function: True
Materials dataset: (120, 14)
Products dataset: (300, 8)
Candidate combinations: 36000
Successful test recommendations: 10
No-result test recommendations: 0

Existing model files:
Cost model: True
CO2 model: True
Preprocessor: True


In [77]:
# Create a safe backup of the existing recommendation function

import inspect

recommend_materials_backup = recommend_materials

print("Original function preserved:", callable(recommend_materials_backup))
print("Backup function name:", recommend_materials_backup.__name__)

# Confirm the original function still works
backup_test = recommend_materials_backup("P0002", top_n=10)

print("Backup test recommendations:", len(backup_test))
print("Original function remains unchanged and available.")

Original function preserved: True
Backup function name: recommend_materials
Backup test recommendations: 10
Original function remains unchanged and available.


In [78]:
def recommend_materials_v2(product_id, top_n=10):

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    product_matches = products_df[
        products_df["product_id"] == product_id
    ]

    if len(product_matches) != 1:
        raise ValueError(
            f"Expected one product for {product_id}, "
            f"but found {len(product_matches)}."
        )

    candidates = candidate_analysis[
        candidate_analysis["product_id"] == product_id
    ].copy()

    result_columns = [
        "rank", "material_id", "material_name",
        "material_category", "predicted_cost_inr",
        "predicted_co2_kg", "biodegradability_score",
        "recyclability_percent", "co2_score",
        "cost_score", "final_score",
        "recommendation_reason"
    ]

    candidates["technical_pass"] = (
        (candidates["strength_score"] >= candidates["required_strength_score"])
        & (candidates["max_weight_capacity_kg"] >= candidates["product_weight_kg"])
    )

    candidates["moisture_pass"] = (
        (candidates["moisture_sensitive"] == 0)
        | (candidates["moisture_resistance_score"] >= MIN_MOISTURE_RESISTANCE)
    )

    candidates["budget_pass"] = (
        candidates["predicted_cost_inr"] <= candidates["budget_per_pack_inr"]
    )

    eligible = candidates[
        candidates["technical_pass"]
        & candidates["moisture_pass"]
        & candidates["budget_pass"]
    ].copy()

    if eligible.empty:
        return pd.DataFrame(columns=result_columns)

    def lower_is_better(series):
        minimum = series.min()
        maximum = series.max()

        if maximum == minimum:
            return pd.Series(100.0, index=series.index)

        return (maximum - series) / (maximum - minimum) * 100

    eligible["co2_score"] = lower_is_better(
        eligible["predicted_co2_kg"]
    )

    eligible["cost_score"] = lower_is_better(
        eligible["predicted_cost_inr"]
    )

    eligible["final_score"] = (
        eligible["co2_score"] * 0.40
        + eligible["biodegradability_score"] * 0.25
        + eligible["recyclability_percent"] * 0.20
        + eligible["cost_score"] * 0.15
    )

    eligible["recommendation_reason"] = (
        "Passed strength, capacity, moisture and budget checks."
    )

    eligible = eligible.sort_values(
        by=["final_score", "predicted_co2_kg", "predicted_cost_inr"],
        ascending=[False, True, True],
        kind="stable"
    ).reset_index(drop=True)

    eligible["rank"] = range(1, len(eligible) + 1)

    return eligible[result_columns].head(top_n)


print("Improved recommendation function created successfully.")

Improved recommendation function created successfully.


In [79]:
# Test 1: Product with eligible recommendations

original_result = recommend_materials_backup("P0002", top_n=10)

v2_result = recommend_materials_v2("P0002", top_n=10)

print("P0002 - Original recommendations:", len(original_result))
print("P0002 - V2 recommendations:", len(v2_result))

# Test 2: Product with no eligible recommendations

original_empty = recommend_materials_backup("P0003", top_n=10)

v2_empty = recommend_materials_v2("P0003", top_n=10)

print("\nP0003 - Original recommendations:", len(original_empty))
print("P0003 - V2 recommendations:", len(v2_empty))

print("\nV2 output columns:")
print(v2_result.columns.tolist())

print("\nV2 empty output columns:")
print(v2_empty.columns.tolist())

# Final checks
assert len(v2_result) == 10
assert len(v2_empty) == 0
assert len(v2_empty.columns) == 12
assert v2_result["rank"].tolist() == list(range(1, 11))

print("\nALL V2 BASIC TESTS PASSED!")

P0002 - Original recommendations: 10
P0002 - V2 recommendations: 10

P0003 - Original recommendations: 0
P0003 - V2 recommendations: 0

V2 output columns:
['rank', 'material_id', 'material_name', 'material_category', 'predicted_cost_inr', 'predicted_co2_kg', 'biodegradability_score', 'recyclability_percent', 'co2_score', 'cost_score', 'final_score', 'recommendation_reason']

V2 empty output columns:
['rank', 'material_id', 'material_name', 'material_category', 'predicted_cost_inr', 'predicted_co2_kg', 'biodegradability_score', 'recyclability_percent', 'co2_score', 'cost_score', 'final_score', 'recommendation_reason']

ALL V2 BASIC TESTS PASSED!


In [80]:
# Compare original and V2 recommendation rankings

comparison = pd.DataFrame({
    "original_material_id": original_result["material_id"].tolist(),
    "v2_material_id": v2_result["material_id"].tolist(),
    "original_score": original_result["final_score"].tolist(),
    "v2_score": v2_result["final_score"].tolist()
})

display(comparison.round(6))

# Verify material ranking consistency
assert (
    comparison["original_material_id"]
    == comparison["v2_material_id"]
).all()

# Verify score consistency
assert np.allclose(
    comparison["original_score"],
    comparison["v2_score"],
    atol=1e-8
)

print("Original and V2 rankings match!")
print("Original and V2 scores match!")

,original_material_id,v2_material_id,original_score,v2_score
0,M0708,M0708,63.784279,63.784279
1,M0705,M0705,63.422163,63.422163
2,M1107,M1107,60.249428,60.249428
3,M1110,M1110,59.709007,59.709007
4,M0807,M0807,53.360683,53.360683
5,M0809,M0809,52.454869,52.454869
6,M0806,M0806,50.031205,50.031205
7,M0805,M0805,45.321696,45.321696
8,M0808,M0808,40.389535,40.389535
9,M0810,M0810,39.259034,39.259034


Original and V2 rankings match!
Original and V2 scores match!
